# Frozen test

The earlier notebooks are exploratory. This one fixes the method first and then runs a single evaluation on operating
points that no earlier notebook used. Fixed in advance: the model revision, the overload margin, the solver tolerance,
starts and iteration caps, the fallback rules, the three outcomes (verified safe, verified violation, unresolved), the
metrics and the rule that sizes the final set. Nothing is tuned afterwards.

Four workflows run on the same cases:

| Workflow | Starts |
|---|---|
| `classical` | pre-contingency state, DC, flat |
| `genco_verify` | GENCO's state first, then the classical starts |
| `classical_first` | the classical starts, GENCO's state only if they fail |
| `classical_recovery` | the classical starts, then a longer iteration cap and a load-stepping homotopy |

End-to-end time per case includes graph construction, transfer to the GPU, inference, verification and classification.
A pilot on the first operating points measures the run time; the sizing rule then fixes the size of the final set before
any of its results are seen.

The verification module is embedded in the notebook. The results in `results/frozen_test` were produced with module
version 0.4.0; later versions added reactive-limit handling and the branch check, which the frozen policy does not use.

Runtime: T4 GPU, about 0.6 GB of downloads, 40 to 60 minutes. Output: `frozen_test.zip`.

## Settings (frozen)

In [ ]:
REPO_ID = "gridfm/genco-pf-contingency"
HF_REVISION = "5c625b00f5b79c8a7d4d27ecc133d4ecdf573af7"       # the revision every run used (unchanged since 1 October 2026)
GRAPHKIT_COMMIT = "95b90eecc7f5c2c86a9dd776f0c3f40434def22f"   # the commit every run used (branch head from 1 to 14 October 2026)
LOCAL_DIR = "genco-pf-contingency"
OUT_DIR = "frozen_test"
GRAPHKIT_DIR = "gridfm-graphkit"
GRAPHKIT_BRANCH = "genco-paper-repro"
NETWORK = "Texas2k_case1_2016summerpeak"
MODEL_DIR = "model/training_bs_16"

LEVEL_BASE = 2
N_PARTS = 2                           # N-2 data blocks to use; chosen below by scanning which blocks hold untouched operating points
SKIP_PARTS = [0, 1]                   # the blocks of the storm / stress-test / construction base points, never used here
MIN_UNTOUCHED_PER_PART = 120          # a block qualifies when it holds at least this many untouched operating points
EVAL_PARTS = None                     # set by the scan
N_CHECK = 16                          # base graphs on which the loaded model is checked
FAMILIES = [("uniform", 10), ("uniform", 20), ("nearest", 10), ("nearest", 20)]   # representative mixture, per base point
CHALLENGE = [("corridor", 20)]        # the deliberately difficult set, reported separately
WORKFLOWS = ["classical", "genco_verify", "classical_first", "classical_recovery"]
POLICY = {"tol_pu": 1e-8, "max_iter": 30, "order": ["genco", "base", "dc", "flat"], "line_limit": 1.0, "voltage_buses": "pq", "low_voltage_flag": 0.5,
          "recovery": {"max_iter": 100, "homotopy_steps": [0.25, 0.5, 0.75, 1.0]}, "screen_margin": 0.0454, "enforce_q_limits": False, "q_limit_rounds": 15}
GATE = {"signal": "resP_p99_MW", "threshold_MW": 0.5414,
        "rule": "fixed before this run: the 10th percentile of GENCO's 99th-percentile active residual over the 2,969 scenarios of notebook 01 (its 'most trusted 10 %')"}
PILOT_BASES = 20
TIME_BUDGET_MIN = 25                  # the final set is sized by the pilot's runtime, never by its results
MAX_BASES = 300
BOOTSTRAP = 1000
THROUGHPUT_BATCHES = [16, 64]
PYPOWER_SAMPLE = 20                   # verified states checked with PYPOWER besides every case resolved only with GENCO
QLIM_SEARCH = 60                      # cases searched for a reactive-limit example for the demo
REGION_HOPS, BETWEENNESS_SAMPLES = 4, 400
BATCH = 1
TOL, MAX_NR = 1e-8, 30
MAX_DOWNLOAD_GB = 3
SEED = 0
EXCLUSIONS_JSON = '{"used_n2_ids": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 533, 534, 535, 536, 537, 538, 539, 540, 541, 542, 543, 544, 545, 546, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558, 559, 560, 561, 562, 563, 564, 565, 566, 567, 568, 569, 570, 571, 572, 573, 574, 575, 576, 577, 578, 579, 580, 581, 582, 583, 584, 585, 586, 587, 588, 589, 590, 591, 592, 593, 594, 595, 596, 597, 598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 657, 658, 659, 660, 661, 662, 663, 664, 665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675, 676, 677, 678, 679, 680, 681, 682, 683, 684, 685, 686, 687, 688, 689, 690, 691, 692, 693, 694, 695, 696, 697, 698, 699, 700, 701, 702, 703, 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 716, 717, 718, 719, 720, 721, 722, 723, 724, 725, 726, 727, 728, 729, 730, 731, 732, 733, 734, 735, 736, 737, 738, 739, 740, 741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766, 767, 768, 769, 770, 771, 772, 773, 774, 775, 777, 778, 779, 780, 781, 782, 783, 784, 785, 786, 787, 788, 789, 790, 791, 792, 793, 794, 795, 796, 797, 798, 799, 800, 801, 802, 803, 804, 805, 806, 807, 808, 809, 810, 811, 812, 813, 814, 815, 816, 817, 818, 819, 820, 821, 822, 823, 824, 825, 826, 827, 828, 829, 830, 832, 833, 834, 835, 836, 837, 838, 839, 840, 842, 843, 844, 845, 846, 847, 848, 849, 850, 851, 852, 853, 854, 855, 856, 857, 858, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 869, 870, 871, 872, 873, 874, 875, 876, 877, 878, 879, 880, 881, 882, 883, 884, 885, 886, 887, 889, 890, 891, 892, 893, 894, 895, 896, 897, 898, 899, 900, 901, 902, 903, 904, 905, 906, 907, 908, 909, 910, 911, 912, 913, 914, 915, 916, 917, 918, 919, 920, 921, 922, 923, 924, 925, 926, 927, 928, 929, 930, 931, 932, 933, 934, 935, 936, 937, 939, 940, 941, 942, 943, 944, 945, 946, 947, 948, 949, 950, 951, 952, 953, 954, 955, 956, 957, 958, 959, 960, 961, 962, 963, 964, 965, 966, 967, 968, 969, 970, 971, 972, 973, 974, 975, 976, 977, 978, 979, 980, 981, 982, 983, 984, 985, 986, 987, 988, 989, 990, 991, 992, 993, 994, 995, 996, 997, 998, 999, 1201, 1203, 1207, 1209, 1214, 1215, 1216, 1217, 1219, 1222, 1223, 1224, 1225, 1226, 1227, 1230, 1231, 1232, 1233, 1235, 1236, 1238, 1241, 1242, 1243, 1247, 1249, 1250, 1251, 1253, 1255, 1256, 1257, 1258, 1259, 1262, 1264, 1265, 1267, 1269, 1270, 1271, 1273, 1275, 1276, 1277, 1278, 1281, 1282, 1284, 1285, 1286, 1291, 1292, 1293, 1294, 1295, 1301, 1302, 1305, 1306, 1308, 1311, 1312, 1315, 1316, 1317, 1318, 1320, 1326, 1328, 1330, 1333, 1334, 1340, 1344, 1345, 1348, 1354, 1356, 1357, 1359, 1361, 1363, 1364, 1367, 1369, 1370, 1371, 1372, 1373, 1380, 1382, 1384, 1385, 1387, 1389, 1390, 1391, 1392, 1393, 1394, 1395, 1396, 1397, 1402, 1404, 1405, 1406, 1407, 1410, 1411, 1412, 1413, 1416, 1418, 1420, 1422, 1424, 1425, 1427, 1429, 1438, 1444, 1445, 1446, 1447, 1448, 1449, 1450, 1451, 1452, 1454, 1455, 1457, 1458, 1459, 1460, 1461, 1463, 1465, 1466, 1468, 1470, 1471, 1476, 1478, 1485, 1486, 1487, 1491, 1492, 1495, 1496, 1497, 1498, 1499, 1501, 1504, 1506, 1509, 1510, 1512, 1513, 1518, 1519, 1520, 1521, 1524, 1525, 1526, 1527, 1528, 1529, 1531, 1534, 1537, 1539, 1540, 1545, 1549, 1551, 1554, 1555, 1556, 1559, 1560, 1562, 1563, 1564, 1567, 1569, 1571, 1573, 1574, 1575, 1576, 1577, 1578, 1579, 1581, 1583, 1584, 1585, 1593, 1594, 1595, 1596, 1598, 2201, 2202, 2204, 2205, 2206, 2207, 2208, 2209, 2210, 2212, 2213, 2214, 2215, 2218, 2219, 2220, 2222, 2223, 2225, 2226, 2228, 2229, 2230, 2231, 2233, 2235, 2236, 2239, 2240, 2242, 2243, 2245, 2246, 2247, 2248, 2249, 2250, 2251, 2252, 2253, 2255, 2257, 2260, 2261, 2262, 2265, 2266, 2267, 2268, 2269, 2271, 2274, 2276, 2278, 2280, 2281, 2282, 2285, 2288, 2289, 2290, 2291, 2292, 2293, 2296, 2297, 2299, 2300, 2301, 2303, 2306, 2307, 2308, 2311, 2315, 2319, 2321, 2323, 2325, 2326, 2327, 2328, 2330, 2332, 2340, 2341, 2344, 2347, 2351, 2352, 2353, 2356, 2358, 2359, 2360, 2361, 2362, 2363, 2364, 2365, 2366, 2367, 2368, 2370, 2371, 2372, 2374, 2375, 2376, 2383, 2386, 2388, 2389, 2390, 2391, 2392, 2393, 2394, 2395, 2398, 2399, 2618, 2627, 2633, 2634, 2641, 2644, 2650, 2651, 2654, 2656, 2658, 2661, 2668, 2672, 2678, 2680, 2683, 2688, 2693, 2694, 2697, 2705, 2712, 2717, 2721, 2727, 2730, 2744, 2749, 2755, 2757, 2761, 2767, 2770, 2775, 2778, 2782, 2788, 3201, 3202, 3205, 3206, 3211, 3212, 3215, 3217, 3218, 3220, 3221, 3223, 3225, 3227, 3229, 3230, 3231, 3234, 3235, 3238, 3240, 3241, 3244, 3245, 3246, 3248, 3249, 3254, 3259, 3260, 3261, 3263, 3265, 3268, 3269, 3272, 3274, 3278, 3279, 3282, 3286, 3287, 3288, 3291, 3292, 3294, 3296, 3297, 3298, 3299, 3300, 3301, 3305, 3312, 3314, 3315, 3316, 3318, 3320, 3321, 3322, 3324, 3325, 3326, 3327, 3329, 3330, 3331, 3333, 3334, 3335, 3336, 3337, 3338, 3339, 3341, 3342, 3343, 3344, 3345, 3352, 3353, 3363, 3368, 3371, 3372, 3373, 3375, 3376, 3377, 3378, 3379, 3380, 3381, 3382, 3384, 3388, 3391, 3393, 3395, 3602, 3607, 3609, 3610, 3629, 3639, 3641, 3646, 3647, 3657, 3662, 3671, 3683, 3684, 3686, 3690, 3692, 3696, 3698, 3699, 3707, 3711, 3722, 3732, 3734, 3738, 3746, 3747, 3748, 3750, 3753, 3758, 3759, 3760, 3765, 3770, 3773, 3774, 3777, 3779, 3780, 3782, 3783, 3784, 3787, 3799, 3800, 3801, 3805, 3808, 3811, 3812, 3814, 3815, 3816, 3818, 3819, 3820, 3821, 3822, 3825, 3826, 3828, 3830, 3831, 3832, 3833, 3837, 3840, 3841, 3842, 3844, 3848, 3851, 3852, 3853, 3854, 3855, 3858, 3859, 3860, 3861, 3862, 3864, 3865, 3867, 3868, 3869, 3870, 3871, 3874, 3875, 3876, 3877, 3879, 3881, 3882, 3883, 3885, 3886, 3888, 3890, 3891, 3892, 3893, 3894, 3895, 3898, 3899, 3900, 3901, 3903, 3904, 3905, 3906, 3907, 3908, 3909, 3911, 3920, 3924, 3925, 3927, 3928, 3929, 3930, 3931, 3933, 3934, 3935, 3936, 3938, 3940, 3942, 3945, 3946, 3947, 3951, 3956, 3957, 3959, 3960, 3962, 3965, 3971, 3975, 3977, 3978, 3979, 3982, 3983, 3984, 3986, 3987, 3989, 3990, 3992, 3993, 3995, 3996, 3997, 3998, 3999, 4600, 4601, 4603, 4604, 4605, 4606, 4607, 4608, 4609, 4611, 4612, 4613, 4614, 4617, 4618, 4619, 4622, 4623, 4624, 4625, 4626, 4627, 4628, 4629, 4630, 4631, 4633, 4634, 4636, 4637, 4638, 4640, 4643, 4644, 4646, 4649, 4652, 4653, 4656, 4657, 4658, 4661, 4662, 4663, 4664, 4666, 4667, 4668, 4670, 4671, 4672, 4673, 4678, 4679, 4680, 4681, 4684, 4686, 4687, 4688, 4690, 4695, 4698, 4701, 4703, 4707, 4709, 4711, 4713, 4716, 4718, 4719, 4721, 4722, 4723, 4724, 4725, 4726, 4727, 4728, 4729, 4730, 4732, 4733, 4734, 4735, 4736, 4738, 4740, 4741, 4744, 4746, 4749, 4752, 4754, 4756, 4758, 4760, 4761, 4762, 4765, 4767, 4769, 4774, 4775, 4776, 4777, 4778, 4779, 4780, 4782, 4783, 4784, 4786, 4787, 4788, 4789, 4790, 4792, 4798, 4799, 4800, 4801, 4802, 4803, 4807, 4808, 4810, 4812, 4813, 4814, 4817, 4818, 4820, 4821, 4823, 4824, 4825, 4827, 4828, 4830, 4831, 4833, 4834, 4835, 4838, 4839, 4840, 4841, 4842, 4847, 4848, 4851, 4853, 4855, 4856, 4857, 4861, 4863, 4864, 4865, 4870, 4871, 4872, 4873, 4874, 4876, 4878, 4880, 4881, 4884, 4886, 4887, 4888, 4889, 4893, 4894, 4896, 4897, 4898, 4899, 4900, 4902, 4907, 4908, 4909, 4910, 4911, 4912, 4915, 4916, 4920, 4922, 4923, 4924, 4926, 4927, 4928, 4929, 4931, 4932, 4934, 4935, 4936, 4937, 4938, 4940, 4942, 4946, 4947, 4950, 4951, 4952, 4954, 4956, 4958, 4961, 4964, 4966, 4967, 4970, 4971, 4975, 4981, 4982, 4983, 4984, 4985, 4986, 4987, 4988, 4990, 4991, 4992, 4993, 4994, 4995, 4998, 4999, 5201, 5204, 5205, 5206, 5210, 5211, 5214, 5215, 5216, 5217, 5218, 5219, 5221, 5222, 5224, 5225, 5226, 5227, 5229, 5230, 5231, 5235, 5237, 5238, 5241, 5244, 5246, 5247, 5249, 5250, 5251, 5253, 5255, 5258, 5260, 5261, 5262, 5263, 5264, 5266, 5269, 5270, 5273, 5274, 5277, 5279, 5280, 5282, 5284, 5285, 5287, 5288, 5289, 5290, 5294, 5295, 5296, 5297, 5299, 5300, 5301, 5303, 5304, 5306, 5308, 5309, 5310, 5313, 5315, 5316, 5317, 5318, 5321, 5322, 5323, 5325, 5326, 5327, 5328, 5330, 5332, 5333, 5334, 5335, 5337, 5339, 5340, 5341, 5345, 5346, 5348, 5349, 5350, 5352, 5353, 5354, 5355, 5356, 5357, 5358, 5359, 5362, 5363, 5364, 5365, 5367, 5368, 5372, 5373, 5374, 5376, 5377, 5379, 5380, 5381, 5382, 5385, 5386, 5390, 5395, 5397, 5399, 5606, 5607, 5608, 5611, 5612, 5614, 5620, 5621, 5622, 5624, 5631, 5636, 5644, 5654, 5658, 5659, 5660, 5664, 5667, 5673, 5675, 5677, 5685, 5692, 5696, 5699, 5704, 5705, 5706, 5708, 5710, 5711, 5713, 5716, 5721, 5726, 5727, 5728, 5729, 5737, 5742, 5750, 5756, 5761, 5764, 5766, 5770, 5784, 5786, 5791, 5800, 5801, 5802, 5803, 5804, 5805, 5806, 5809, 5810, 5811, 5812, 5813, 5816, 5817, 5818, 5820, 5821, 5824, 5827, 5828, 5829, 5830, 5831, 5834, 5835, 5836, 5837, 5838, 5840, 5844, 5845, 5847, 5848, 5849, 5851, 5853, 5854, 5855, 5856, 5858, 5859, 5860, 5862, 5863, 5865, 5867, 5868, 5869, 5873, 5874, 5877, 5880, 5881, 5882, 5884, 5888, 5890, 5891, 5896, 5897, 5902, 5903, 5905, 5906, 5907, 5908, 5909, 5912, 5914, 5917, 5918, 5919, 5920, 5923, 5925, 5927, 5931, 5932, 5933, 5934, 5938, 5941, 5944, 5945, 5946, 5948, 5949, 5950, 5952, 5953, 5954, 5955, 5956, 5959, 5960, 5961, 5962, 5964, 5966, 5967, 5968, 5969, 5971, 5973, 5974, 5975, 5977, 5978, 5979, 5981, 5982, 5986, 5988, 5989, 5992, 5993, 5994, 5996, 5998, 5999, 6000, 6001, 6009, 6013, 6016, 6017, 6018, 6019, 6020, 6023, 6025, 6027, 6028, 6029, 6030, 6033, 6041, 6042, 6045, 6046, 6047, 6048, 6054, 6055, 6056, 6057, 6058, 6060, 6062, 6063, 6065, 6067, 6069, 6071, 6072, 6073, 6074, 6075, 6078, 6080, 6081, 6082, 6083, 6084, 6087, 6088, 6089, 6094, 6096, 6099, 6103, 6104, 6106, 6107, 6108, 6109, 6110, 6112, 6113, 6114, 6116, 6117, 6118, 6119, 6120, 6124, 6125, 6126, 6132, 6133, 6135, 6139, 6140, 6141, 6144, 6145, 6146, 6148, 6150, 6151, 6152, 6155, 6160, 6165, 6166, 6168, 6169, 6171, 6173, 6174, 6175, 6176, 6177, 6180, 6181, 6182, 6183, 6184, 6185, 6186, 6188, 6193, 6195, 6196, 6197, 6199, 6200, 6201, 6202, 6203, 6204, 6205, 6206, 6208, 6209, 6211, 6212, 6214, 6217, 6219, 6222, 6225, 6226, 6227, 6230, 6232, 6233, 6236, 6237, 6238, 6239, 6241, 6244, 6246, 6249, 6250, 6251, 6253, 6255, 6256, 6258, 6259, 6260, 6262, 6267, 6270, 6272, 6273, 6275, 6277, 6278, 6280, 6282, 6285, 6286, 6290, 6292, 6294, 6296, 6297, 6298, 6299, 6300, 6301, 6302, 6307, 6308, 6310, 6311, 6312, 6313, 6314, 6315, 6317, 6318, 6319, 6320, 6321, 6323, 6324, 6325, 6326, 6327, 6328, 6329, 6330, 6331, 6333, 6334, 6335, 6336, 6338, 6339, 6341, 6342, 6343, 6344, 6348, 6349, 6351, 6355, 6356, 6357, 6358, 6359, 6361, 6362, 6363, 6364, 6367, 6369, 6371, 6375, 6378, 6379, 6380, 6381, 6382, 6386, 6387, 6388, 6389, 6390, 6391, 6394, 6395, 6398, 6399, 7200, 7201, 7202, 7205, 7206, 7212, 7213, 7214, 7216, 7217, 7219, 7220, 7221, 7222, 7223, 7224, 7226, 7227, 7228, 7229, 7230, 7231, 7233, 7234, 7238, 7242, 7243, 7247, 7248, 7249, 7250, 7251, 7252, 7253, 7254, 7255, 7256, 7257, 7259, 7261, 7262, 7263, 7264, 7265, 7266, 7267, 7269, 7271, 7273, 7277, 7278, 7279, 7280, 7285, 7287, 7289, 7290, 7291, 7292, 7293, 7295, 7298, 7299, 7304, 7307, 7308, 7309, 7310, 7313, 7317, 7318, 7321, 7322, 7325, 7329, 7330, 7332, 7334, 7336, 7339, 7342, 7343, 7345, 7346, 7349, 7350, 7351, 7353, 7355, 7356, 7358, 7359, 7363, 7364, 7366, 7368, 7371, 7372, 7373, 7374, 7378, 7382, 7384, 7385, 7387, 7388, 7390, 7391, 7393, 7394, 7395, 7396, 7397, 7398, 7399, 7600, 7604, 7605, 7606, 7607, 7611, 7612, 7614, 7615, 7618, 7622, 7623, 7624, 7626, 7627, 7634, 7635, 7637, 7638, 7639, 7640, 7642, 7643, 7645, 7647, 7650, 7651, 7652, 7654, 7658, 7659, 7661, 7663, 7664, 7665, 7667, 7668, 7671, 7672, 7675, 7676, 7677, 7680, 7684, 7685, 7686, 7687, 7688, 7689, 7691, 7693, 7696, 7697, 7701, 7704, 7705, 7708, 7711, 7715, 7718, 7719, 7722, 7723, 7725, 7726, 7732, 7734, 7735, 7736, 7738, 7740, 7741, 7742, 7743, 7744, 7746, 7748, 7753, 7754, 7755, 7756, 7758, 7759, 7761, 7762, 7763, 7764, 7765, 7766, 7767, 7769, 7770, 7775, 7778, 7779, 7781, 7782, 7783, 7784, 7785, 7786, 7789, 7790, 7791, 7792, 7793, 7794, 7796, 7797, 7798, 7799, 7803, 7805, 7806, 7809, 7811, 7812, 7813, 7814, 7816, 7818, 7820, 7823, 7824, 7828, 7830, 7833, 7835, 7837, 7842, 7843, 7844, 7847, 7848, 7849, 7851, 7853, 7854, 7855, 7858, 7859, 7861, 7863, 7865, 7866, 7868, 7869, 7870, 7871, 7872, 7874, 7875, 7882, 7883, 7885, 7886, 7888, 7889, 7892, 7894, 7895, 7896, 7898, 7899, 7900, 7901, 7902, 7907, 7908, 7909, 7911, 7912, 7913, 7914, 7915, 7917, 7918, 7919, 7920, 7922, 7923, 7924, 7925, 7926, 7928, 7930, 7931, 7933, 7935, 7936, 7937, 7938, 7941, 7943, 7944, 7945, 7946, 7947, 7949, 7950, 7951, 7952, 7954, 7955, 7956, 7957, 7959, 7960, 7961, 7962, 7963, 7964, 7966, 7967, 7968, 7969, 7970, 7971, 7974, 7975, 7976, 7977, 7978, 7979, 7980, 7981, 7982, 7983, 7985, 7989, 7994, 7995, 7996, 7997, 7998, 8000, 8001, 8002, 8003, 8004, 8006, 8008, 8016, 8017, 8018, 8019, 8022, 8025, 8026, 8030, 8032, 8035, 8037, 8039, 8041, 8047, 8049, 8050, 8052, 8054, 8058, 8060, 8063, 8067, 8069, 8070, 8071, 8072, 8073, 8074, 8075, 8078, 8080, 8082, 8083, 8085, 8088, 8089, 8090, 8091, 8092, 8099, 8101, 8103, 8104, 8105, 8107, 8108, 8110, 8111, 8112, 8113, 8115, 8116, 8119, 8121, 8123, 8125, 8126, 8129, 8130, 8137, 8138, 8140, 8141, 8142, 8143, 8144, 8146, 8147, 8148, 8149, 8150, 8151, 8153, 8155, 8157, 8158, 8159, 8160, 8163, 8164, 8166, 8167, 8168, 8170, 8171, 8177, 8180, 8184, 8185, 8186, 8187, 8188, 8189, 8190, 8191, 8192, 8193, 8198, 8200, 8202, 8203, 8206, 8207, 8210, 8214, 8217, 8218, 8219, 8220, 8221, 8222, 8224, 8226, 8230, 8232, 8234, 8235, 8236, 8237, 8239, 8242, 8244, 8248, 8250, 8253, 8254, 8255, 8257, 8260, 8261, 8262, 8267, 8268, 8270, 8271, 8273, 8275, 8276, 8277, 8279, 8281, 8283, 8285, 8286, 8287, 8288, 8289, 8291, 8292, 8293, 8294, 8295, 8296, 8298, 8299, 8303, 8304, 8307, 8309, 8311, 8312, 8313, 8314, 8316, 8317, 8318, 8322, 8323, 8324, 8327, 8328, 8330, 8331, 8333, 8336, 8337, 8338, 8339, 8340, 8341, 8342, 8343, 8346, 8349, 8350, 8351, 8352, 8353, 8354, 8356, 8357, 8358, 8360, 8362, 8363, 8364, 8365, 8366, 8372, 8374, 8375, 8380, 8381, 8383, 8385, 8386, 8387, 8391, 8394, 8399, 8400, 8403, 8405, 8406, 8407, 8408, 8409, 8411, 8414, 8415, 8417, 8418, 8419, 8420, 8421, 8422, 8424, 8425, 8426, 8427, 8428, 8429, 8430, 8431, 8435, 8436, 8439, 8440, 8441, 8442, 8443, 8444, 8445, 8447, 8450, 8451, 8453, 8455, 8458, 8460, 8461, 8463, 8464, 8465, 8469, 8470, 8472, 8474, 8476, 8480, 8483, 8485, 8487, 8488, 8491, 8492, 8493, 8494, 8495, 8496, 8497, 8499, 8500, 8502, 8503, 8505, 8506, 8508, 8510, 8513, 8514, 8515, 8516, 8517, 8518, 8519, 8520, 8522, 8524, 8525, 8527, 8528, 8530, 8533, 8534, 8539, 8540, 8542, 8546, 8549, 8558, 8559, 8562, 8565, 8567, 8568, 8570, 8572, 8575, 8576, 8577, 8579, 8580, 8582, 8584, 8586, 8587, 8588, 8592, 8593, 8594, 8595], "n2_test_role_ids": [5, 6, 7, 8, 10, 16, 18, 19, 20, 21, 23, 24, 26, 30, 32, 33, 35, 37, 38, 41, 42, 44, 45, 48, 54, 56, 57, 58, 59, 60, 65, 66, 68, 71, 73, 75, 78, 80, 83, 90, 91, 93, 94, 95, 96, 97, 99, 101, 102, 103, 104, 107, 108, 110, 113, 116, 118, 119, 120, 121, 122, 123, 124, 125, 128, 129, 130, 138, 141, 142, 144, 145, 148, 149, 150, 151, 152, 154, 156, 157, 159, 160, 161, 164, 165, 166, 167, 175, 176, 178, 180, 183, 188, 189, 191, 194, 196, 197, 199, 402, 408, 411, 412, 413, 414, 415, 417, 418, 419, 422, 424, 429, 435, 436, 437, 438, 443, 445, 447, 451, 457, 458, 462, 470, 471, 472, 473, 474, 475, 477, 479, 481, 489, 493, 496, 497, 498, 499, 502, 505, 507, 508, 509, 511, 512, 513, 514, 515, 517, 518, 519, 522, 523, 524, 525, 527, 530, 535, 536, 538, 539, 542, 543, 544, 545, 546, 548, 549, 553, 555, 556, 557, 558, 567, 570, 571, 573, 574, 575, 576, 577, 578, 579, 581, 582, 584, 588, 590, 591, 592, 593, 596, 598, 600, 605, 608, 610, 611, 612, 614, 618, 621, 622, 623, 625, 629, 631, 634, 635, 636, 638, 641, 643, 645, 651, 653, 655, 657, 658, 660, 661, 663, 664, 667, 669, 670, 673, 674, 679, 680, 683, 684, 685, 687, 688, 689, 691, 692, 694, 697, 698, 699, 705, 706, 708, 709, 711, 714, 716, 718, 729, 730, 731, 734, 736, 738, 741, 742, 743, 744, 745, 746, 747, 752, 753, 754, 756, 757, 758, 759, 763, 765, 767, 768, 770, 778, 779, 791, 792, 794, 797, 799, 1200, 1202, 1204, 1205, 1206, 1208, 1210, 1211, 1212, 1213, 1216, 1218, 1220, 1221, 1228, 1229, 1231, 1234, 1237, 1239, 1240, 1244, 1245, 1246, 1248, 1249, 1252, 1254, 1260, 1261, 1263, 1264, 1266, 1268, 1272, 1274, 1279, 1280, 1282, 1283, 1284, 1287, 1288, 1289, 1290, 1294, 1296, 1297, 1298, 1299, 1300, 1303, 1304, 1307, 1309, 1310, 1311, 1313, 1314, 1319, 1321, 1322, 1323, 1324, 1325, 1327, 1329, 1331, 1332, 1335, 1336, 1337, 1338, 1339, 1341, 1342, 1343, 1346, 1347, 1349, 1350, 1351, 1352, 1353, 1355, 1358, 1360, 1362, 1365, 1366, 1368, 1374, 1375, 1376, 1377, 1378, 1379, 1381, 1383, 1385, 1386, 1388, 1396, 1397, 1398, 1399, 1400, 1401, 1403, 1408, 1409, 1414, 1415, 1417, 1419, 1421, 1423, 1426, 1428, 1430, 1431, 1432, 1433, 1434, 1435, 1436, 1437, 1439, 1440, 1441, 1442, 1443, 1452, 1453, 1456, 1457, 1462, 1464, 1467, 1469, 1472, 1473, 1474, 1475, 1477, 1479, 1481, 1482, 1483, 1484, 1488, 1489, 1490, 1492, 1493, 1494, 1499, 1500, 1502, 1503, 1505, 1507, 1508, 1511, 1514, 1515, 1516, 1517, 1521, 1522, 1523, 1530, 1532, 1533, 1535, 1536, 1538, 1541, 1542, 1543, 1544, 1546, 1547, 1548, 1550, 1551, 1552, 1553, 1557, 1558, 1561, 1565, 1566, 1568, 1569, 1570, 1572, 1574, 1575, 1580, 1582, 1586, 1587, 1588, 1589, 1590, 1591, 1592, 1597, 1599, 2200, 2202, 2203, 2204, 2206, 2207, 2209, 2210, 2211, 2215, 2216, 2217, 2221, 2222, 2224, 2225, 2226, 2227, 2229, 2230, 2232, 2234, 2236, 2237, 2238, 2241, 2242, 2244, 2247, 2250, 2252, 2254, 2256, 2258, 2259, 2263, 2264, 2266, 2268, 2270, 2271, 2272, 2273, 2275, 2276, 2277, 2279, 2282, 2283, 2284, 2286, 2287, 2288, 2294, 2295, 2296, 2298, 2300, 2302, 2304, 2305, 2309, 2310, 2312, 2313, 2314, 2316, 2317, 2318, 2320, 2322, 2324, 2327, 2329, 2331, 2333, 2334, 2335, 2336, 2337, 2338, 2339, 2340, 2342, 2343, 2345, 2346, 2348, 2349, 2350, 2352, 2353, 2354, 2355, 2357, 2360, 2364, 2367, 2369, 2372, 2373, 2377, 2378, 2379, 2380, 2381, 2382, 2383, 2384, 2385, 2387, 2390, 2391, 2394, 2396, 2397, 2399, 3200, 3203, 3204, 3205, 3207, 3208, 3209, 3210, 3214, 3216, 3217, 3219, 3222, 3224, 3226, 3228, 3232, 3233, 3236, 3237, 3239, 3242, 3243, 3247, 3249, 3250, 3251, 3252, 3253, 3254, 3255, 3256, 3257, 3258, 3262, 3263, 3264, 3266, 3267, 3269, 3270, 3271, 3273, 3274, 3275, 3276, 3277, 3279, 3280, 3281, 3283, 3284, 3285, 3289, 3290, 3293, 3295, 3296, 3302, 3303, 3304, 3306, 3307, 3308, 3309, 3310, 3311, 3313, 3314, 3317, 3319, 3323, 3327, 3328, 3332, 3339, 3340, 3341, 3346, 3347, 3348, 3349, 3350, 3351, 3354, 3355, 3356, 3357, 3358, 3359, 3360, 3361, 3362, 3364, 3365, 3366, 3367, 3369, 3370, 3374, 3379, 3382, 3383, 3385, 3386, 3387, 3389, 3390, 3392, 3394, 3396, 3397, 3398, 3399, 3802, 3803, 3804, 3806, 3807, 3809, 3810, 3813, 3817, 3823, 3824, 3827, 3829, 3834, 3835, 3836, 3838, 3839, 3843, 3844, 3845, 3846, 3847, 3849, 3850, 3852, 3856, 3857, 3863, 3866, 3872, 3873, 3875, 3878, 3880, 3881, 3884, 3887, 3889, 3892, 3896, 3897, 3898, 3902, 3905, 3910, 3912, 3913, 3914, 3915, 3916, 3917, 3918, 3919, 3920, 3921, 3922, 3923, 3925, 3926, 3927, 3928, 3930, 3932, 3937, 3939, 3941, 3943, 3944, 3947, 3948, 3949, 3950, 3952, 3953, 3954, 3955, 3958, 3961, 3963, 3964, 3966, 3967, 3968, 3969, 3970, 3972, 3973, 3974, 3976, 3980, 3981, 3982, 3985, 3988, 3991, 3994, 3997, 4602, 4608, 4609, 4610, 4615, 4616, 4619, 4620, 4621, 4632, 4635, 4638, 4639, 4641, 4642, 4645, 4647, 4648, 4650, 4651, 4654, 4655, 4659, 4660, 4661, 4665, 4669, 4673, 4674, 4675, 4676, 4677, 4682, 4683, 4684, 4685, 4689, 4691, 4692, 4693, 4694, 4696, 4697, 4699, 4700, 4702, 4704, 4705, 4706, 4708, 4710, 4711, 4712, 4713, 4714, 4715, 4717, 4720, 4723, 4731, 4737, 4739, 4742, 4743, 4745, 4747, 4748, 4750, 4751, 4753, 4755, 4757, 4759, 4763, 4764, 4766, 4768, 4770, 4771, 4772, 4773, 4779, 4781, 4785, 4790, 4791, 4793, 4794, 4795, 4796, 4797, 4804, 4805, 4806, 4807, 4809, 4811, 4815, 4816, 4819, 4822, 4826, 4829, 4832, 4836, 4837, 4842, 4843, 4844, 4845, 4846, 4849, 4850, 4852, 4854, 4858, 4859, 4860, 4862, 4866, 4867, 4868, 4869, 4875, 4877, 4879, 4882, 4883, 4885, 4890, 4891, 4892, 4894, 4895, 4898, 4901, 4902, 4903, 4904, 4905, 4906, 4913, 4914, 4917, 4918, 4919, 4921, 4923, 4924, 4925, 4930, 4933, 4936, 4939, 4941, 4943, 4944, 4945, 4948, 4949, 4953, 4955, 4957, 4959, 4960, 4962, 4963, 4965, 4968, 4969, 4970, 4972, 4973, 4974, 4976, 4977, 4978, 4979, 4980, 4987, 4989, 4992, 4996, 4997, 4999, 5200, 5201, 5202, 5203, 5207, 5208, 5209, 5212, 5213, 5215, 5216, 5220, 5223, 5228, 5231, 5232, 5233, 5234, 5236, 5237, 5239, 5240, 5242, 5243, 5245, 5248, 5252, 5254, 5256, 5257, 5258, 5259, 5265, 5267, 5268, 5271, 5272, 5275, 5276, 5278, 5279, 5281, 5283, 5286, 5287, 5291, 5292, 5293, 5296, 5298, 5302, 5305, 5307, 5309, 5311, 5312, 5314, 5319, 5320, 5324, 5327, 5329, 5331, 5332, 5336, 5338, 5342, 5343, 5344, 5347, 5350, 5351, 5353, 5360, 5361, 5366, 5368, 5369, 5370, 5371, 5375, 5378, 5379, 5383, 5384, 5387, 5388, 5389, 5391, 5392, 5393, 5394, 5396, 5397, 5398, 5804, 5807, 5808, 5814, 5815, 5819, 5820, 5822, 5823, 5824, 5825, 5826, 5832, 5833, 5838, 5839, 5841, 5842, 5843, 5846, 5850, 5852, 5854, 5857, 5861, 5864, 5865, 5866, 5867, 5868, 5870, 5871, 5872, 5875, 5876, 5877, 5878, 5879, 5883, 5885, 5886, 5887, 5889, 5890, 5892, 5893, 5894, 5895, 5897, 5898, 5899, 5900, 5901, 5904, 5908, 5910, 5911, 5912, 5913, 5915, 5916, 5921, 5922, 5924, 5928, 5929, 5930, 5931, 5933, 5935, 5936, 5937, 5938, 5939, 5940, 5941, 5942, 5943, 5945, 5947, 5949, 5951, 5953, 5957, 5958, 5963, 5965, 5968, 5970, 5972, 5973, 5975, 5976, 5978, 5980, 5982, 5983, 5984, 5985, 5987, 5990, 5991, 5992, 5995, 5997, 6002, 6003, 6004, 6005, 6006, 6007, 6008, 6010, 6011, 6012, 6014, 6015, 6021, 6022, 6024, 6026, 6029, 6031, 6032, 6034, 6035, 6036, 6037, 6038, 6039, 6040, 6043, 6044, 6046, 6049, 6050, 6051, 6052, 6053, 6055, 6057, 6059, 6061, 6063, 6064, 6066, 6068, 6070, 6076, 6077, 6079, 6085, 6086, 6089, 6090, 6091, 6092, 6093, 6097, 6098, 6100, 6101, 6102, 6104, 6105, 6109, 6111, 6115, 6121, 6122, 6123, 6127, 6128, 6129, 6130, 6131, 6134, 6135, 6136, 6137, 6138, 6142, 6143, 6147, 6149, 6151, 6153, 6154, 6156, 6157, 6158, 6159, 6161, 6162, 6163, 6164, 6167, 6168, 6170, 6172, 6176, 6178, 6179, 6187, 6189, 6190, 6191, 6192, 6194, 6198, 6200, 6207, 6208, 6210, 6213, 6215, 6216, 6218, 6220, 6221, 6223, 6224, 6228, 6229, 6231, 6234, 6235, 6240, 6242, 6243, 6245, 6247, 6248, 6252, 6254, 6256, 6257, 6261, 6263, 6264, 6265, 6266, 6268, 6269, 6271, 6273, 6274, 6275, 6276, 6279, 6280, 6281, 6283, 6284, 6286, 6287, 6288, 6289, 6291, 6293, 6294, 6295, 6303, 6304, 6305, 6306, 6309, 6316, 6319, 6320, 6322, 6324, 6332, 6337, 6340, 6345, 6346, 6347, 6350, 6352, 6353, 6354, 6356, 6360, 6365, 6366, 6367, 6368, 6370, 6372, 6373, 6374, 6377, 6383, 6384, 6385, 6386, 6388, 6392, 6393, 6396, 6397, 7203, 7204, 7207, 7208, 7209, 7210, 7211, 7215, 7218, 7219, 7222, 7223, 7225, 7228, 7232, 7235, 7236, 7237, 7239, 7240, 7241, 7244, 7245, 7246, 7248, 7250, 7252, 7257, 7258, 7260, 7262, 7264, 7268, 7270, 7272, 7274, 7275, 7276, 7281, 7282, 7283, 7284, 7286, 7287, 7288, 7294, 7296, 7297, 7300, 7301, 7302, 7303, 7305, 7306, 7311, 7312, 7314, 7315, 7316, 7319, 7320, 7323, 7324, 7325, 7326, 7327, 7328, 7331, 7333, 7334, 7335, 7337, 7338, 7340, 7341, 7344, 7347, 7348, 7352, 7354, 7357, 7360, 7361, 7362, 7365, 7367, 7368, 7369, 7370, 7375, 7376, 7377, 7379, 7380, 7381, 7383, 7386, 7389, 7392, 7397, 7398, 7601, 7602, 7603, 7608, 7609, 7610, 7611, 7613, 7616, 7617, 7619, 7620, 7621, 7625, 7627, 7628, 7629, 7630, 7631, 7632, 7633, 7636, 7641, 7644, 7646, 7648, 7649, 7652, 7653, 7655, 7656, 7657, 7658, 7660, 7662, 7666, 7669, 7670, 7673, 7674, 7675, 7676, 7678, 7679, 7681, 7682, 7683, 7684, 7690, 7692, 7693, 7694, 7695, 7698, 7699, 7700, 7702, 7703, 7706, 7707, 7709, 7710, 7711, 7712, 7713, 7714, 7716, 7717, 7720, 7721, 7724, 7727, 7728, 7729, 7730, 7731, 7733, 7737, 7739, 7740, 7744, 7745, 7747, 7749, 7750, 7751, 7752, 7757, 7760, 7763, 7768, 7771, 7772, 7773, 7774, 7775, 7776, 7777, 7780, 7787, 7788, 7790, 7795, 7800, 7801, 7802, 7804, 7807, 7808, 7810, 7815, 7816, 7817, 7819, 7821, 7822, 7824, 7825, 7826, 7827, 7829, 7830, 7831, 7832, 7833, 7834, 7836, 7838, 7839, 7840, 7841, 7843, 7844, 7845, 7846, 7850, 7852, 7856, 7857, 7860, 7862, 7863, 7864, 7867, 7870, 7873, 7876, 7877, 7878, 7879, 7880, 7881, 7882, 7884, 7886, 7887, 7890, 7891, 7893, 7897, 7903, 7904, 7905, 7906, 7910, 7916, 7918, 7920, 7921, 7923, 7927, 7929, 7932, 7934, 7939, 7940, 7942, 7948, 7953, 7955, 7958, 7962, 7965, 7972, 7973, 7974, 7984, 7986, 7987, 7988, 7990, 7991, 7992, 7993, 7999, 8005, 8007, 8008, 8009, 8010, 8011, 8012, 8013, 8014, 8015, 8020, 8021, 8023, 8024, 8027, 8028, 8029, 8031, 8033, 8034, 8036, 8038, 8040, 8042, 8043, 8044, 8045, 8046, 8048, 8051, 8053, 8055, 8056, 8057, 8059, 8061, 8062, 8064, 8065, 8066, 8068, 8077, 8079, 8081, 8084, 8086, 8087, 8093, 8094, 8095, 8096, 8097, 8098, 8099, 8100, 8101, 8102, 8106, 8109, 8114, 8117, 8118, 8120, 8122, 8123, 8124, 8127, 8128, 8131, 8132, 8133, 8134, 8135, 8136, 8139, 8145, 8147, 8152, 8154, 8155, 8156, 8161, 8162, 8165, 8169, 8172, 8173, 8174, 8175, 8176, 8178, 8179, 8181, 8182, 8183, 8191, 8194, 8195, 8196, 8197, 8199, 8201, 8204, 8205, 8208, 8209, 8211, 8212, 8213, 8215, 8216, 8218, 8223, 8225, 8227, 8228, 8229, 8231, 8233, 8238, 8240, 8241, 8243, 8245, 8246, 8247, 8248, 8249, 8251, 8252, 8255, 8256, 8258, 8259, 8263, 8264, 8265, 8266, 8268, 8269, 8272, 8274, 8278, 8280, 8282, 8284, 8297, 8299, 8300, 8301, 8302, 8303, 8305, 8306, 8308, 8310, 8315, 8319, 8320, 8321, 8325, 8326, 8329, 8331, 8332, 8334, 8335, 8344, 8345, 8347, 8348, 8355, 8359, 8361, 8363, 8367, 8368, 8369, 8370, 8371, 8372, 8373, 8376, 8377, 8378, 8379, 8382, 8384, 8388, 8389, 8390, 8392, 8393, 8395, 8396, 8397, 8398, 8400, 8401, 8402, 8404, 8410, 8412, 8413, 8416, 8423, 8431, 8432, 8433, 8434, 8437, 8438, 8439, 8446, 8448, 8449, 8452, 8454, 8456, 8457, 8459, 8462, 8466, 8467, 8468, 8471, 8473, 8475, 8477, 8478, 8479, 8481, 8482, 8483, 8484, 8486, 8489, 8490, 8498, 8501, 8504, 8507, 8509, 8511, 8512, 8521, 8522, 8523, 8526, 8529, 8531, 8532, 8535, 8536, 8537, 8538, 8541, 8542, 8543, 8544, 8545, 8547, 8548, 8550, 8551, 8552, 8553, 8554, 8555, 8556, 8557, 8559, 8560, 8561, 8563, 8564, 8565, 8566, 8569, 8571, 8573, 8574, 8578, 8581, 8582, 8583, 8585, 8589, 8590, 8591, 8596, 8597, 8598, 8599]}'

In [ ]:
# [GPU] install and import

import importlib.util, os, subprocess, sys, time


def run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")


import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Use Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.\n"
          "The notebook also works on CPU, but the timing results would not be meaningful.")

if not os.path.isdir(GRAPHKIT_DIR):
    run(f"git clone -q -b {GRAPHKIT_BRANCH} https://github.com/gridfm/gridfm-graphkit.git {GRAPHKIT_DIR}")
    run(f"git -C {GRAPHKIT_DIR} checkout -q {GRAPHKIT_COMMIT}")   # pinned: the commit every run used
need = {"torch_geometric": "torch_geometric", "lightning": "lightning", "pytorch_lightning": "pytorch-lightning",
        "huggingface_hub": "huggingface_hub", "pyarrow": "pyarrow", "seaborn": "seaborn", "sklearn": "scikit-learn"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    run(f"{sys.executable} -m pip install -q " + " ".join(missing))

import torch_geometric   # imported before the replacement below, so PyTorch Geometric keeps its own native code paths
USED_SHIM = importlib.util.find_spec("torch_scatter") is None
if USED_SHIM:
    shim = os.path.abspath(os.path.join("shims", "torch_scatter"))
    os.makedirs(shim, exist_ok=True)
    with open(os.path.join(shim, "__init__.py"), "w") as f:
        f.write('"""Small pure-PyTorch stand-in for torch_scatter (only the functions gridfm-graphkit uses)."""\nimport torch\n\n__version__ = "0.0.0-shim"\n\n\ndef _prep(src, index, dim):\n    dim = dim if dim >= 0 else src.dim() + dim\n    if index.dim() != src.dim():\n        shape = [1] * src.dim()\n        shape[dim] = -1\n        index = index.view(shape).expand_as(src)\n    return index, dim\n\n\ndef _size(src, index, dim, dim_size):\n    size = list(src.shape)\n    if dim_size is None:\n        dim_size = int(index.max()) + 1 if index.numel() > 0 else 0\n    size[dim] = dim_size\n    return size\n\n\ndef scatter_add(src, index, dim=-1, out=None, dim_size=None):\n    index, dim = _prep(src, index, dim)\n    if out is None:\n        out = torch.zeros(_size(src, index, dim, dim_size), dtype=src.dtype, device=src.device)\n    return out.scatter_add_(dim, index, src)\n\n\nscatter_sum = scatter_add\n\n\ndef scatter_mean(src, index, dim=-1, out=None, dim_size=None):\n    total = scatter_add(src, index, dim, None, dim_size)\n    idx, d = _prep(src, index, dim)\n    count = torch.zeros_like(total).scatter_add_(d, idx, torch.ones_like(src)).clamp_(min=1)\n    return total / count\n\n\ndef scatter_max(src, index, dim=-1, out=None, dim_size=None):\n    idx, d = _prep(src, index, dim)\n    res = torch.full(_size(src, idx, d, dim_size), float("-inf"), dtype=src.dtype, device=src.device)\n    res = res.scatter_reduce(d, idx, src, reduce="amax", include_self=True)\n    res[torch.isinf(res)] = 0\n    return res, None\n\n\ndef scatter(src, index, dim=-1, out=None, dim_size=None, reduce="sum"):\n    if reduce in ("sum", "add"):\n        return scatter_add(src, index, dim, out, dim_size)\n    if reduce == "mean":\n        return scatter_mean(src, index, dim, out, dim_size)\n    if reduce == "max":\n        return scatter_max(src, index, dim, out, dim_size)[0]\n    raise NotImplementedError(reduce)\n')
    sys.path.insert(0, os.path.abspath("shims"))
    print("torch_scatter is not installed: using a small built-in replacement")
sys.path.insert(0, os.path.abspath(GRAPHKIT_DIR))

import gridfm_graphkit
from gridfm_graphkit.io.param_handler import NestedNamespace, load_normalizer, get_task_transforms, get_task
from gridfm_graphkit.training.callbacks import adapt_state_dict_for_model
from gridfm_graphkit.datasets.hetero_preprocess import build_hetero_data_for_scenario, merge_agg_gen_into_bus
from gridfm_graphkit.datasets.globals import VM_OUT, VA_OUT
from torch_geometric.loader import DataLoader
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch_geometric", torch_geometric.__version__, "| gridfm-graphkit from", os.path.abspath(GRAPHKIT_DIR), "| device:", DEVICE)

In [ ]:
# lightsim2grid: open-source C++ Newton-Raphson (KLU sparse solver), used here as the professional AC solver
import importlib.util, subprocess, sys
if importlib.util.find_spec("lightsim2grid") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightsim2grid"], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
print("lightsim2grid installed:", importlib.util.find_spec("lightsim2grid") is not None)

In [ ]:
# PYPOWER (the Python port of MATPOWER): the independent implementation used for the validation
import importlib.util, subprocess, sys
if importlib.util.find_spec("pypower") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypower"], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError("pip install pypower failed: " + r.stderr[-800:])
import pypower
from importlib.metadata import version as _pkg_version
print("PYPOWER", _pkg_version("PYPOWER"))

In [ ]:
import base64
GENCO_VERIFY_SOURCE = base64.b64decode('...').decode()
TEST_SOURCE = base64.b64decode('...').decode()
exec(GENCO_VERIFY_SOURCE)
print('genco_verify loaded; lightsim2grid (KLU):', HAVE_KLU)

In [ ]:
# ===================== core functions =====================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import splu

SN_MVA = 100.0
BUS_COLS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF",
            "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc", "vn_kv"]
BR_COLS = ["scenario", "idx", "from_bus", "to_bus", "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i",
           "Ytt_r", "Ytt_i", "rate_a", "br_status", "pf_dc", "pt_dc"]
PRED_ALIASES = {"vm": ["vm_pu", "Vm_pred", "VM_pred", "vm_pred"], "va": ["va", "Va_pred", "VA_pred", "va_pred"]}


# ---------------------------------------------------------------- file helpers
def find_local(local_dir, k, table):
    base = Path(local_dir) / "data" / f"test_{k}"
    hits = sorted(base.rglob(f"{table}.parquet")) if base.exists() else []
    if not hits:
        raise FileNotFoundError(f"No {table}.parquet under {base}")
    return hits[0]


def find_pred(local_dir, k):
    base = Path(local_dir) / "mlflow" / "eval" / f"test_{k}"
    hits = sorted(base.rglob("*predictions.parquet")) if base.exists() else []
    hits = [h for h in hits if not re.search(r"_(gen|branch)_predictions\.parquet$", h.name)]
    if not hits:
        raise FileNotFoundError(f"No predictions.parquet under {base}")
    return hits[0]


def _dataset(path):
    return ds.dataset(str(path), format="parquet", partitioning="hive")


def read_cols(path, wanted, scen=None):
    dset = _dataset(path)
    cols = [c for c in wanted if c in dset.schema.names]
    flt = None
    if scen is not None:
        values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
        flt = ds.field("scenario").isin(values)
    return dset.to_table(columns=cols, filter=flt).to_pandas()


def scenario_ids(path):
    return np.unique(_dataset(path).to_table(columns=["scenario"]).column("scenario").to_numpy().astype(np.int64))


# ---------------------------------------------------------------- one chunk of scenarios
def load_chunk(bus_path, br_path, pred_chunk, scen):
    """Arrays for a chunk of scenarios. pred_chunk: DataFrame with scenario, bus, vm, va (already filtered)."""
    bus = read_cols(bus_path, BUS_COLS, scen=scen)
    bus["scenario"] = bus["scenario"].astype(np.int64)
    bus = bus.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    pred = pred_chunk.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    S = len(scen)
    nb = len(bus) // S
    if len(bus) != S * nb or len(pred) != S * nb:
        raise RuntimeError(f"rows do not match: bus {len(bus)}, predictions {len(pred)}, {S} scenarios")
    ids = np.broadcast_to(np.arange(nb), (S, nb))
    if not (np.array_equal(bus["bus"].to_numpy().reshape(S, nb), ids) and np.array_equal(pred["bus"].to_numpy().reshape(S, nb), ids)):
        raise RuntimeError("bus numbers are not 0..nb-1 in every scenario")
    A = {c: bus[c].to_numpy(np.float64).reshape(S, nb) for c in BUS_COLS[2:] if c in bus.columns}
    A["PQ"], A["PV"], A["REF"] = A["PQ"] > 0.5, A["PV"] > 0.5, A["REF"] > 0.5
    A["Vm_g"] = pred["vm"].to_numpy(np.float64).reshape(S, nb)
    va = pred["va"].to_numpy(np.float64).reshape(S, nb)
    rad_err = np.nanmedian(np.abs(np.degrees(va) - A["Va"]))
    deg_err = np.nanmedian(np.abs(va - A["Va"]))
    A["va_unit"] = "rad" if rad_err <= deg_err else "deg"
    A["Va_g"] = np.degrees(va) if A["va_unit"] == "rad" else va
    br = read_cols(br_path, BR_COLS, scen=scen)
    A["sc"] = np.searchsorted(scen, br["scenario"].to_numpy().astype(np.int64))
    A["line"] = (br["idx"].to_numpy().astype(np.int64) if "idx" in br.columns
                 else br.groupby("scenario").cumcount().to_numpy().astype(np.int64))
    A["f"] = br["from_bus"].to_numpy().astype(np.int64)
    A["t"] = br["to_bus"].to_numpy().astype(np.int64)
    for y in ["Yff", "Yft", "Ytf", "Ytt"]:
        A[y] = br[y + "_r"].to_numpy(np.float64) + 1j * br[y + "_i"].to_numpy(np.float64)
    A["rate"] = br["rate_a"].to_numpy(np.float64)
    A["on"] = (br["br_status"].to_numpy(np.float64) > 0.5) if "br_status" in br.columns else (np.abs(A["Yff"]) > 0)
    pdc = br["pf_dc"].to_numpy(np.float64) if "pf_dc" in br.columns else np.full(len(br), np.nan)
    ptdc = br["pt_dc"].to_numpy(np.float64) if "pt_dc" in br.columns else pdc
    A["P_dc_abs"] = np.maximum(np.abs(pdc), np.abs(ptdc))
    A["S"], A["nb"], A["scen"] = S, nb, np.asarray(scen)
    return A


def branch_flows(A, V):
    sc, f, t = A["sc"], A["f"], A["t"]
    Vf, Vt = V[sc, f], V[sc, t]
    Sf = Vf * np.conj(A["Yff"] * Vf + A["Yft"] * Vt) * SN_MVA
    St = Vt * np.conj(A["Ytt"] * Vt + A["Ytf"] * Vf) * SN_MVA
    return Sf, St


def bus_mismatch(A, Vm, Sf, St):
    S, nb, sc, f, t = A["S"], A["nb"], A["sc"], A["f"], A["t"]
    add = lambda v, at: np.bincount(sc * nb + at, weights=v, minlength=S * nb).reshape(S, nb)
    P = (A["Pg"] - A["Pd"]) - (add(Sf.real, f) + add(St.real, t)) - A["GS"] * Vm**2 * SN_MVA
    Q = (A["Qg"] - A["Qd"]) - (add(Sf.imag, f) + add(St.imag, t)) + A["BS"] * Vm**2 * SN_MVA
    return P, Q


def _scen_max(values, sc, mask, S):
    out = np.full(S, np.nan)
    v = np.where(mask, values, -np.inf)
    tmp = np.full(S, -np.inf); np.maximum.at(tmp, sc, v)
    out[np.isfinite(tmp)] = tmp[np.isfinite(tmp)]
    return out


def chunk_tables(A, near_line=0.4, near_v=0.08):
    """Compact line-level and bus-level tables plus per-scenario signals for one chunk."""
    S, nb = A["S"], A["nb"]
    V_t = A["Vm"] * np.exp(1j * np.radians(A["Va"]))
    V_g = A["Vm_g"] * np.exp(1j * np.radians(A["Va_g"]))
    V_d = np.exp(1j * np.radians(A["Va_dc"]))
    Sf_t, St_t = branch_flows(A, V_t)
    Sf_g, St_g = branch_flows(A, V_g)
    Sf_d, St_d = branch_flows(A, V_d)
    P_t, _ = bus_mismatch(A, A["Vm"], Sf_t, St_t)
    P_g, Q_g = bus_mismatch(A, A["Vm_g"], Sf_g, St_g)
    resP = np.where(A["REF"], np.nan, np.abs(P_g))
    with np.errstate(all="ignore"):
        r_p99 = np.nanpercentile(resP, 99, axis=1)
    rated = A["on"] & (A["rate"] > 0)
    rate = np.where(rated, A["rate"], np.nan)
    ld = lambda Sf, St: np.maximum(np.abs(Sf), np.abs(St)) / rate
    ld_t, ld_g, ld_dS = ld(Sf_t, St_t), ld(Sf_g, St_g), ld(Sf_d, St_d)
    ld_dP = A["P_dc_abs"] / rate
    dc_failed = np.isnan(A["Va_dc"]).any(axis=1)
    keep = rated & ((ld_t >= near_line) | (ld_g >= near_line) | (np.nan_to_num(ld_dS) >= near_line) | (np.nan_to_num(ld_dP) >= near_line))
    lines = pd.DataFrame({
        "sidx": A["sc"][keep].astype(np.int32), "line": A["line"][keep].astype(np.int32),
        "ld_true": ld_t[keep].astype(np.float32), "ld_genco": ld_g[keep].astype(np.float32),
        "ld_dc_paper": ld_dS[keep].astype(np.float32), "ld_dc_p": ld_dP[keep].astype(np.float32),
    })
    lines.loc[dc_failed[lines.sidx.to_numpy()], ["ld_dc_paper", "ld_dc_p"]] = np.nan
    # PQ buses close to a voltage limit (truth or GENCO)
    PQ, vmin, vmax = A["PQ"], A["min_vm_pu"], A["max_vm_pu"]
    dist_t = np.minimum(A["Vm"] - vmin, vmax - A["Vm"])
    dist_g = np.minimum(A["Vm_g"] - vmin, vmax - A["Vm_g"])
    kb = PQ & ((dist_t <= near_v) | (dist_g <= near_v))
    si, bi = np.nonzero(kb)
    buses = pd.DataFrame({"sidx": si.astype(np.int32), "bus": bi.astype(np.int32),
                          "vm_true": A["Vm"][kb].astype(np.float32), "vm_genco": A["Vm_g"][kb].astype(np.float32),
                          "vmin": vmin[kb].astype(np.float32), "vmax": vmax[kb].astype(np.float32)})
    scen = pd.DataFrame({
        "scenario": A["scen"], "res_p99": r_p99,
        "n_lines_out": np.bincount(A["sc"][~A["on"]], minlength=S),
        "n_overload_true": np.bincount(A["sc"][rated], weights=(ld_t[rated] >= 1.0), minlength=S),
        "n_vviol_true": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax))).sum(axis=1),
        "err_vm_max": np.nanmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), np.nan), axis=1),
        "worst_bus": np.nanargmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), -1.0), axis=1),
        "err_load_max": _scen_max(np.abs(ld_g - ld_t), A["sc"], rated, S),
        "n_missed_overload_genco": np.bincount(A["sc"][rated], weights=((ld_t[rated] >= 1.0) & (ld_g[rated] < 1.0)), minlength=S),
        "n_missed_vviol_genco": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax)) & (A["Vm_g"] >= vmin) & (A["Vm_g"] <= vmax)).sum(axis=1),
        "dc_failed": dc_failed,
    })
    scen["worst_bus_kv"] = A["vn_kv"][np.arange(S), scen["worst_bus"].to_numpy()] if "vn_kv" in A else np.nan
    checks = {"truth_mismatch_max_MW": float(np.nanmax(np.abs(np.where(A["REF"], 0, P_t)))),
              "va_unit": A["va_unit"]}
    return lines, buses, scen, checks


# ---------------------------------------------------------------- margins
def fit_exponent(e, r, n_bins=8, q=0.99, min_count=200):
    """Slope of log(quantile of error) against log(scenario residual), from bins of the residual. Clipped to [0, 2]."""
    e, r = np.asarray(e, float), np.asarray(r, float)
    ok = np.isfinite(e) & np.isfinite(r) & (r > 0)
    e, r = e[ok], r[ok]
    if len(e) < 3 * min_count:
        return 0.0, []
    if len(e) < n_bins * min_count:
        n_bins = max(3, len(e) // min_count)
    edges = np.unique(np.quantile(r, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, r, side="right") - 1, 0, len(edges) - 2)
    xs, ys = [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() >= min_count:
            qq = np.quantile(e[m], q)
            if qq > 0:
                xs.append(np.log(np.median(r[m]))); ys.append(np.log(qq))
    if len(xs) < 3:
        return 0.0, list(zip(xs, ys))
    return float(np.clip(np.polyfit(xs, ys, 1)[0], 0.0, 2.0)), list(zip(xs, ys))


def scale_of(r, r_ref, b):
    return np.power(np.maximum(np.asarray(r, float), 1e-9) / r_ref, b)


# ---------------------------------------------------------------- screening curves
def screening_curve(score, positive, sidx_global, n_scen, thresholds):
    """Flag an item if score >= threshold. Returns recall, false alarms per scenario, share of scenarios with a flag."""
    score = np.asarray(score, float); positive = np.asarray(positive, bool)
    ok = np.isfinite(score)
    score, positive, sidx = score[ok], positive[ok], np.asarray(sidx_global)[ok]
    n_pos = max(int(positive.sum()), 1)
    order = np.argsort(-score, kind="stable")
    s_desc = -score[order]                          # ascending
    cum_pos = np.r_[0, np.cumsum(positive[order])]
    cum_neg = np.r_[0, np.cumsum(~positive[order])]
    smax = np.full(n_scen, -np.inf); np.maximum.at(smax, sidx, score)
    smax_sorted = np.sort(smax)
    th = np.asarray(thresholds, float)
    n = np.searchsorted(s_desc, -th, side="right")   # items with score >= th
    tp, fp = cum_pos[n], cum_neg[n]
    sent = n_scen - np.searchsorted(smax_sorted, th, side="left")
    return pd.DataFrame({"threshold": th, "recall": tp / n_pos, "misses": n_pos - tp,
                         "false_alarms_per_scenario": fp / n_scen, "share_scenarios_flagged": sent / n_scen})


# ---------------------------------------------------------------- Newton-Raphson (polar, MATPOWER formulation)
def build_ybus(nb, f, t, Yff, Yft, Ytf, Ytt, GS, BS):
    rows = np.concatenate([f, f, t, t, np.arange(nb)])
    cols = np.concatenate([f, t, f, t, np.arange(nb)])
    vals = np.concatenate([Yff, Yft, Ytf, Ytt, GS + 1j * BS])
    return sp.csr_matrix((vals, (rows, cols)), shape=(nb, nb))


def dsbus_dv(Ybus, V):
    Ibus = Ybus @ V
    dV, dI, dVn = sp.diags(V), sp.diags(Ibus), sp.diags(V / np.abs(V))
    dS_dVm = dV @ (Ybus @ dVn).conj() + dI.conj() @ dVn
    dS_dVa = 1j * dV @ (dI - Ybus @ dV).conj()
    return dS_dVm.tocsr(), dS_dVa.tocsr()


def newton(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=20, n_steps=None):
    """Returns final V, iterations done, list of max mismatch (p.u.) after each iteration, first-step correction size."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]
    n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V); norms = [float(np.max(np.abs(F)))]; it = 0; first_step = np.nan
    while True:
        if n_steps is None and (norms[-1] <= tol or it >= max_it):
            break
        if n_steps is not None and it >= n_steps:
            break
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            norms.append(np.inf); break
        if it == 0:
            first_step = float(np.max(np.abs(dx)))
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V); it += 1; norms.append(float(np.max(np.abs(F))))
        if not np.isfinite(norms[-1]) or norms[-1] > 1e6:
            break
    return V, it, norms, first_step


def scenario_system(A, i):
    """Ybus, specified power and bus sets for scenario i of a chunk."""
    nb = A["nb"]
    m = (A["sc"] == i) & A["on"]
    Y = build_ybus(nb, A["f"][m], A["t"][m], A["Yff"][m], A["Yft"][m], A["Ytf"][m], A["Ytt"][m], A["GS"][i], A["BS"][i])
    Sspec = (A["Pg"][i] - A["Pd"][i] + 1j * (A["Qg"][i] - A["Qd"][i])) / SN_MVA
    pv, pq = np.nonzero(A["PV"][i])[0], np.nonzero(A["PQ"][i])[0]
    return Y, Sspec, pv, pq


def extract_scenario(A, i):
    """Keep only what Newton needs for scenario i (small, so it can be stored)."""
    m = A["sc"] == i
    out = {key: A[key][i].copy() for key in ["Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "GS", "BS",
                                             "Va_dc", "Vm_g", "Va_g", "min_vm_pu", "max_vm_pu", "vn_kv"] if key in A}
    for key in ["f", "t", "Yff", "Yft", "Ytf", "Ytt", "rate", "on", "line"]:
        if key in A:
            out[key] = A[key][m].copy()
    out["nb"] = A["nb"]; out["scenario"] = int(A["scen"][i])
    return out


def run_newton_cases(d, tol=1e-8, max_it=20):
    """Flat start, DC start, GENCO start (to convergence) and GENCO + exactly one step, for one stored scenario."""
    nb = d["nb"]
    on = d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq, ref = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0], np.nonzero(d["REF"])[0]
    Vm_set = np.where(d["PQ"], 1.0, d["Vm"])          # setpoints at PV and slack, 1.0 elsewhere
    Va_ref = np.where(d["REF"], d["Va"], 0.0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def start(Vm, Va_deg):
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(Vm_set, Va_ref), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    res = {"scenario": d["scenario"]}
    for name, V0 in starts.items():
        t0 = time.perf_counter()
        V, it, norms, step1 = newton(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        res[f"seconds_{name}"] = time.perf_counter() - t0
        conv = norms[-1] <= tol
        res[f"iters_{name}"] = it if conv else np.nan
        res[f"converged_{name}"] = bool(conv)
        res[f"start_mismatch_{name}"] = norms[0]
        if conv:
            res[f"max_dV_vs_truth_{name}"] = float(np.max(np.abs(V - V_true)))
        if name == "genco":
            res["first_step_size"] = step1
    t0 = time.perf_counter()
    V1, _, norms1, _ = newton(Y, Sspec, starts["genco"], pv, pq, n_steps=1)
    res["seconds_genco_1step"] = time.perf_counter() - t0
    res["mismatch_after_1_step"] = norms1[-1]
    rated = on & (d["rate"] > 0)

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))
    for name, V in [("genco", starts["genco"]), ("genco_1step", V1)]:
        ld = loading(V); Vm = np.abs(V)
        vio = d["PQ"] & ((Vm < d["min_vm_pu"]) | (Vm > d["max_vm_pu"]))
        res[f"err_vm_max_{name}"] = float(np.max(np.abs(Vm - d["Vm"])[d["PQ"]])) if d["PQ"].any() else np.nan
        res[f"err_load_max_{name}"] = float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan
        res[f"missed_overloads_{name}"] = int((over_t & (ld < 1.0)).sum())
        res[f"missed_vviol_{name}"] = int((vio_t & ~vio).sum())
    res["true_overloads"] = int(over_t.sum()); res["true_vviol"] = int(vio_t.sum())
    return res


def newton_trace(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=10):
    """List of (V, max mismatch p.u., seconds for this step) after 0, 1, 2, ... Newton steps."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]; n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V)
    out = [(V.copy(), float(np.max(np.abs(F))), 0.0)]
    for _ in range(max_it):
        if out[-1][1] <= tol:
            break
        t0 = time.perf_counter()
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            break
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V)
        out.append((V.copy(), float(np.max(np.abs(F))), time.perf_counter() - t0))
        if not np.isfinite(out[-1][1]) or out[-1][1] > 1e6:
            break
    return out


def run_trace_cases(d, tol=1e-8, max_it=10, line_margin=0.0, v_margin=0.0):
    """Decision metrics after every Newton step from flat, DC and GENCO starts, for one stored scenario.
    Columns ending in _m use the safety margins (flag a line from 1 - line_margin, a bus within v_margin of a limit)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    PQ, lo, hi = d["PQ"], d["min_vm_pu"], d["max_vm_pu"]
    rated = on & (d["rate"] > 0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = PQ & ((d["Vm"] < lo) | (d["Vm"] > hi))

    def start(Vm, Va_deg):
        Vm = np.where(PQ, Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(np.ones(nb), np.zeros(nb)), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    rows = []
    for name, V0 in starts.items():
        trace = newton_trace(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        converged = trace[-1][1] <= tol
        for j, (V, mism, secs) in enumerate(trace):
            ld, Vm = loading(V), np.abs(V)
            vio = PQ & ((Vm < lo) | (Vm > hi))
            vio_m = PQ & ((Vm < lo + v_margin) | (Vm > hi - v_margin))
            fl_m = ld >= 1.0 - line_margin
            rows.append({"scenario": d["scenario"], "start": name, "steps": j, "mismatch": mism, "seconds_step": secs,
                         "converged_at_end": converged, "iterations_to_converge": len(trace) - 1 if converged else np.nan,
                         "err_vm_max": float(np.max(np.abs(Vm - d["Vm"])[PQ])) if PQ.any() else np.nan,
                         "err_load_max": float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan,
                         "missed_overloads": int((over_t & (ld < 1.0)).sum()), "false_overloads": int((~over_t & (ld >= 1.0)).sum()),
                         "missed_vviol": int((vio_t & ~vio).sum()), "false_vviol": int((~vio_t & vio).sum()),
                         "missed_overloads_m": int((over_t & ~fl_m).sum()), "false_overloads_m": int((~over_t & fl_m).sum()),
                         "missed_vviol_m": int((vio_t & ~vio_m).sum()), "false_vviol_m": int((~vio_t & vio_m).sum()),
                         "true_overloads": int(over_t.sum()), "true_vviol": int(vio_t.sum())})
    return rows


# ===================== GENCO helpers =====================
from scipy.sparse.csgraph import dijkstra


class ScenarioEval:
    """Truth-based checks of any voltage state for one stored scenario (from extract_scenario)."""

    def __init__(self, d):
        self.d = d
        nb, on = d["nb"], d["on"]
        self.Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
        self.Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
        self.pv, self.pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.rated = on & (d["rate"] > 0)
        Vt = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
        self.ld_t = self.loading(Vt)
        self.over_t = self.ld_t >= 1.0
        self.vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / d["rate"][self.rated]

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def metrics(self, Vm, Va_deg):
        d = self.d
        V = self.state(Vm, Va_deg)
        S = V * np.conj(self.Y @ V) - self.Sspec
        Pm, Qm = np.abs(S.real), np.abs(S.imag)
        ld = self.loading(V)
        Vabs = np.abs(V)
        vio = d["PQ"] & ((Vabs < d["min_vm_pu"]) | (Vabs > d["max_vm_pu"]))
        err = np.abs(Vabs - d["Vm"])
        return {"err_vm_max": float(err[d["PQ"]].max()) if d["PQ"].any() else np.nan,
                "err_load_max": float(np.max(np.abs(ld - self.ld_t))) if len(ld) else np.nan,
                "mismatch_pu": float(max(Pm[np.r_[self.pv, self.pq]].max(), Qm[self.pq].max())),
                "res_p99_MW": float(np.percentile(Pm[self.nonref], 99) * SN_MVA),
                "missed_overloads": int((self.over_t & (ld < 1.0)).sum()),
                "missed_vviol": int((self.vio_t & ~vio).sum())}

    def bus_errors(self, Vm):
        return np.where(self.d["PQ"], np.abs(Vm - self.d["Vm"]), np.nan)


def thevenin_pq(d, block=200):
    """|Z_ii| for every PQ bus with PV and slack buses held at fixed voltage: how much the voltage at a bus
    moves per unit of current injected there (large = electrically weak spot)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    pq = np.nonzero(d["PQ"])[0]
    lu = splu(Y[pq][:, pq].tocsc())
    n = len(pq); z = np.empty(n, complex)
    for s in range(0, n, block):
        idx = np.arange(s, min(n, s + block))
        E = np.zeros((n, len(idx)), complex); E[idx, np.arange(len(idx))] = 1.0
        X = lu.solve(E)
        z[idx] = X[idx, np.arange(len(idx))]
    return pq, np.abs(z)


def hops_to_voltage_control(d):
    """Number of in-service lines between each bus and the nearest PV or slack bus."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    src = np.nonzero(d["PV"] | d["REF"])[0]
    dist = dijkstra(A, directed=False, unweighted=True, indices=src, min_only=True)
    return dist


def dc_solve_seconds(d, reps=3):
    """Time of one DC power flow (one sparse solve on the susceptance matrix), for the cost comparison."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    keep = np.nonzero(~d["REF"])[0]
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    t0 = time.perf_counter()
    for _ in range(reps):
        B = (-Y.imag)[keep][:, keep].tocsc()
        splu(B).solve(P)
    return (time.perf_counter() - t0) / reps


# ===================== outage construction =====================
from scipy.sparse.csgraph import connected_components
try:
    from lightsim2grid.algorithm import NR_KLU as _KLU
    HAVE_KLU = True
except Exception:
    try:
        from lightsim2grid.solver import KLUSolver as _KLU
        HAVE_KLU = True
    except Exception:
        HAVE_KLU = False


def system_of(d):
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    return Y, S, np.nonzero(d["REF"])[0], np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]


def klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol=1e-8):
    # lightsim2grid's C++ Newton-Raphson with the KLU sparse solver. Returns V, converged, iterations, seconds.
    w = np.zeros(Y.shape[0]); w[ref] = 1.0 / len(ref)
    Yc = Y.tocsc()
    s = _KLU()
    t0 = time.perf_counter()
    ok = s.solve(Yc, np.asarray(V0, complex), np.asarray(S, complex), ref.astype(np.int32), w,
                 pv.astype(np.int32), pq.astype(np.int32), int(max_iter), float(tol))
    dt = time.perf_counter() - t0
    V = np.array(s.get_V(), dtype=complex, copy=True)
    return V, bool(ok), int(s.get_nb_iter()), dt


def dc_angles(d):
    # DC power flow on the current topology (series susceptances, no shunts, no phase shifts). Returns angles in degrees.
    nb, on = d["nb"], d["on"]
    f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
    B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
    ref = np.nonzero(d["REF"])[0]
    keep = np.setdiff1d(np.arange(nb), ref)
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    th = np.zeros(nb)
    th[keep] = splu(B[keep][:, keep]).solve(P)
    return np.degrees(th) + float(d["Va"][ref[0]])


def connected(nb, f, t):
    A = sp.csr_matrix((np.ones(len(f)), (f, t)), shape=(nb, nb))
    return connected_components(A, directed=False)[0] == 1


def pick_outages(d, extra, mode, rng):
    # Indices (into the scenario's branch rows) of `extra` more lines to remove, keeping the grid in one piece.
    # mode "clustered": the lines nearest (in hops) to a random centre bus, like a storm footprint.
    # mode "uniform": random lines anywhere, like the released test sets.
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    hop = None
    if mode == "clustered":
        center = int(rng.integers(nb))
        A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
        hop = dijkstra(A, directed=False, unweighted=True, indices=center)
        dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    else:
        order = rng.permutation(cand)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]]))) if hop is not None else np.nan
    return np.array(removed), radius


def solve_new_topology(d, removed, tol=1e-8, max_iter=30):
    # Exact AC solution after removing `removed`; starts: the base solution, then flat, then DC.
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    d2["on"][removed] = False
    Y, S, ref, pv, pq = system_of(d2)
    base = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    flat = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(np.where(d2["REF"], d2["Va"], 0.0)))
    d2["Va_dc"] = dc_angles(d2)
    dc = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(d2["Va_dc"]))
    for name, V0 in [("base", base), ("flat", flat), ("dc", dc)]:
        if HAVE_KLU:
            V, ok, it, _ = klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol)
        else:
            V, it, norms, _ = newton(Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
            ok = norms[-1] <= tol
        if ok:
            d2["Vm"], d2["Va"] = np.abs(V), np.degrees(np.angle(V))
            d2["solved_from"] = name
            return d2
    return None




"""AC sensitivity helpers: true AC power-flow voltage sensitivities from the Jacobian, a finite-difference check of them,
hop distances in the grid graph, and the comparison metrics. Uses build_ybus / dsbus_dv / newton / SN_MVA from
the shared power-flow helpers above."""
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


def ac_system(d):
    """Ybus, true complex voltages, specified injections (p.u.) and bus sets of one stored scenario."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    V = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    return Y, V, S, pv, pq


def power_flow_jacobian(Y, V, pv, pq):
    """Jacobian of [P mismatch at pv+pq; Q mismatch at pq] with respect to [angle at pv+pq; magnitude at pq]."""
    pvpq = np.r_[pv, pq]
    dS_dVm, dS_dVa = dsbus_dv(Y, V)
    return sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                      sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc"), pvpq


def true_sensitivity_rows(d, targets):
    """For each target bus (PQ in this scenario): dVm_target / dPd_j and dVm_target / dQd_j for every bus j,
    in p.u. voltage per p.u. load (100 MW or 100 Mvar). A load increase is a lower injection, hence the minus sign.
    Loads at the slack bus (P) and at voltage-controlled buses (Q) do not move any voltage: those entries are 0."""
    Y, V, S, pv, pq = ac_system(d)
    J, pvpq = power_flow_jacobian(Y, V, pv, pq)
    lu = splu(J.T.tocsc())
    n1, nb = len(pvpq), d["nb"]
    pos_pq = {int(b): n1 + i for i, b in enumerate(pq)}
    out = {}
    for t in targets:
        if int(t) not in pos_pq:
            continue
        e = np.zeros(J.shape[0]); e[pos_pq[int(t)]] = 1.0
        y = lu.solve(e)
        TP, TQ = np.zeros(nb), np.zeros(nb)
        TP[pvpq] = -y[:n1]
        TQ[pq] = -y[n1:]
        out[int(t)] = (TP, TQ)
    return out


def ac_fd_check(d, t, delta_pu=0.01):
    """Finite-difference check of the Jacobian sensitivities. Converge the base case, raise the reactive load at
    bus t by delta_pu (1 Mvar for 0.01), converge again, and compare dVm_t / dQd_t from the two solutions with the
    Jacobian value at the base solution. Returns (finite difference, Jacobian)."""
    Y, V, S, pv, pq = ac_system(d)
    V0, _, n0, _ = newton(Y, S, V, pv, pq, tol=1e-11, max_it=20)
    S2 = S.copy(); S2[t] -= 1j * delta_pu
    V2, _, n2, _ = newton(Y, S2, V0, pv, pq, tol=1e-11, max_it=20)
    if not n0 or not n2 or n0[-1] > 1e-9 or n2[-1] > 1e-9:
        return np.nan, np.nan
    J, pvpq = power_flow_jacobian(Y, V0, pv, pq)
    row = np.nonzero(pq == t)[0]
    e = np.zeros(J.shape[0]); e[len(pvpq) + int(row[0])] = 1.0
    jac = float(-splu(J.T.tocsc()).solve(e)[len(pvpq) + int(row[0])])
    return float((np.abs(V2[t]) - np.abs(V0[t])) / delta_pu), jac


def hop_distances(d, targets):
    """Number of in-service lines between each target and every bus (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    f, t = d["f"][on], d["t"][on]
    A = sp.csr_matrix((np.ones(2 * len(f)), (np.r_[f, t], np.r_[t, f])), shape=(nb, nb))
    return shortest_path(A, unweighted=True, directed=False, indices=np.asarray(targets, dtype=np.int64))


def row_metrics(gP, gQ, tP, tQ, pq_mask, pv_mask):
    """Compare GENCO's sensitivity row with the true one (same units). Only loads that can move voltages enter the
    gain and cosine: P at every non-slack bus, Q at PQ buses. Leakage: share of GENCO's reactive-load sensitivity that
    sits on voltage-controlled buses, where the true value is exactly 0."""
    g = np.r_[gP[pq_mask | pv_mask], gQ[pq_mask]]
    tr = np.r_[tP[pq_mask | pv_mask], tQ[pq_mask]]
    tt = float(tr @ tr)
    gain = float(g @ tr / tt) if tt > 0 else np.nan
    cos = float(g @ tr / (np.linalg.norm(g) * np.sqrt(tt))) if tt > 0 and np.linalg.norm(g) > 0 else np.nan
    rel_err = float(np.linalg.norm(g - tr) / np.sqrt(tt)) if tt > 0 else np.nan
    leak = float(np.abs(gQ[~pq_mask]).sum() / max(np.abs(gQ).sum(), 1e-30))
    gq, tq = gQ[pq_mask], tQ[pq_mask]
    tqq = float(tq @ tq)
    gain_q = float(gq @ tq / tqq) if tqq > 0 else np.nan
    cos_q = float(gq @ tq / (np.linalg.norm(gq) * np.sqrt(tqq))) if tqq > 0 and np.linalg.norm(gq) > 0 else np.nan
    return {"gain": gain, "cosine": cos, "relative_error": rel_err, "gain_Q": gain_q, "cosine_Q": cos_q, "pv_q_leakage": leak}


def hop_profile(abs_row, hops, max_hop=8):
    """Sum of |sensitivity| at each hop distance 0..max_hop (the last bin holds max_hop and beyond)."""
    h = np.where(np.isfinite(hops), np.minimum(hops, max_hop), max_hop).astype(int)
    return np.bincount(h, weights=abs_row, minlength=max_hop + 1)[: max_hop + 1]


def slope_through(x, y):
    """Least-squares slope of y on x (both centred): how strongly GENCO's voltage follows the true one."""
    x = np.asarray(x, float) - np.mean(x); y = np.asarray(y, float) - np.mean(y)
    return float(x @ y / (x @ x)) if x @ x > 0 else np.nan


"""Load-sweep helpers: one storm topology with every load and every non-slack generator scaled by a load factor lambda (the
slack takes the rest), the AC solution along lambda up to the largest factor at which a Newton start converges (the
"nose"), a check on that nose, and the screening decisions GENCO, DC and the Newton check make at
every lambda. Uses build_ybus / dsbus_dv / newton / newton_trace / klu_solve / system_of / dc_angles / SN_MVA /
HAVE_KLU from the shared power-flow helpers above and power_flow_jacobian from the AC sensitivity helpers."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


class LambdaCase:
    """One topology; injections scaled by lambda: P at every non-slack bus = lambda * (Pg - Pd), Q at PQ buses =
    Qg - lambda * Qd (constant power factor), generator voltage set-points unchanged, the slack takes the rest."""

    def __init__(self, d):
        self.d = d
        self.Y, _, self.ref, self.pv, self.pq = system_of(d)
        self.nb = d["nb"]
        self.PQ, self.PV, self.REF = d["PQ"], d["PV"], d["REF"]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.P0 = (d["Pg"] - d["Pd"]) / SN_MVA
        self.Qg, self.Qd = d["Qg"] / SN_MVA, d["Qd"] / SN_MVA
        self.rated = d["on"] & (d["rate"] > 0)
        self.rate = d["rate"][self.rated]
        self.vmin, self.vmax = d["min_vm_pu"], d["max_vm_pu"]
        Vm_set = np.where(d["PQ"], 1.0, d["Vm"])
        self.flat = Vm_set * np.exp(1j * np.radians(np.where(d["REF"], d["Va"], 0.0)))
        self.dc = Vm_set * np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"])))
        self.ld_dc1 = self.loading(np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"]))))   # DC state, AC branch equations

    # ---------------------------------------------------------------- physics at a given lambda
    def S(self, lam):
        return lam * self.P0 + 1j * (self.Qg - lam * self.Qd)

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def solve(self, lam, V0, max_iter=30, tol=1e-8):
        """Newton-Raphson from V0 at this lambda. Returns V, converged, iterations, seconds."""
        S = self.S(lam)
        if HAVE_KLU:
            return klu_solve(self.Y, S, V0, self.ref, self.pv, self.pq, max_iter, tol)
        t0 = time.perf_counter()
        V, it, norms, _ = newton(self.Y, S, V0, self.pv, self.pq, tol=tol, max_it=max_iter)
        return V, bool(norms[-1] <= tol), it, time.perf_counter() - t0

    def solve_any(self, lam, starts, max_iter=30, tol=1e-8):
        """First converged solution over a list of (name, V0) starts; (None, None) if none converges."""
        for name, V0 in starts:
            V, ok, _, _ = self.solve(lam, V0, max_iter, tol)
            if ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2:
                return V, name
        return None, None

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / self.rate

    def dc_loading(self, lam):
        """DC power flow at this lambda: DC angles scale with lambda, flows through the AC branch equations."""
        return self.loading(np.exp(1j * np.radians(lam * np.nan_to_num(self.d["Va_dc"]))))

    def mismatch(self, lam, V):
        """Active mismatch (MW) at non-slack buses and reactive mismatch (Mvar) at PQ buses of a voltage state."""
        S = (V * np.conj(self.Y @ V) - self.S(lam)) * SN_MVA
        return np.abs(S.real[self.nonref]), np.abs(S.imag[self.pq])

    def jacobian(self, V):
        return power_flow_jacobian(self.Y, V, self.pv, self.pq)[0]

    # ---------------------------------------------------------------- what a screen would say about a state
    def decisions(self, lam, V, margin_line=0.0, margin_v=0.0):
        Vm = np.abs(V)
        ld = self.loading(V)
        P, Q = self.mismatch(lam, V)
        low = self.PQ & (Vm < self.vmin + margin_v)
        high = self.PQ & (Vm > self.vmax - margin_v)
        pq_vm = np.where(self.PQ, Vm, np.inf)
        return {"min_vm": float(pq_vm.min()), "weak_bus": int(np.argmin(pq_vm)),
                "n_low_v": int(low.sum()), "n_high_v": int(high.sum()), "n_over": int((ld >= 1.0 - margin_line).sum()),
                "max_loading": float(ld.max()) if len(ld) else np.nan,
                "resP_max_MW": float(P.max()), "resP_p99_MW": float(np.percentile(P, 99)), "resP_mean_MW": float(P.mean()),
                "resQ_max_Mvar": float(Q.max()) if len(Q) else np.nan,
                "mismatch_pu": float(max(P.max(), Q.max() if len(Q) else 0.0) / SN_MVA)}


# ---------------------------------------------------------------- the nose
def find_lambda_max(case, V_at_one=None, step=0.02, width=0.0025, lam_min=0.5, lam_cap=2.0, tol=1e-8, max_iter=30):
    """Largest load factor with an AC solution, by marching up from a solvable point and bisecting the first failure.
    At every failure the flat and DC starts are tried too, so a failure of the continuation start alone does not count.
    Returns a dict: lam_max, V_max (solution there), lam_first (first solvable point found), path [(lam, who)],
    n_solves, capped (True if solvable up to lam_cap)."""
    path, n = [], 0
    lam, V = 1.0, V_at_one
    if V is None:                       # no start converges at lambda = 1: go down until one does
        lam = 1.0 - step
        while lam >= lam_min:
            n += 3
            V, who = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc), ("base", case.state(case.d["Vm"], case.d["Va"]))], max_iter, tol)
            if V is not None:
                path.append((lam, who)); break
            lam = round(lam - step, 6)
        if V is None:
            return {"lam_max": np.nan, "V_max": None, "lam_first": np.nan, "path": path, "n_solves": n, "capped": False}
    else:
        path.append((1.0, "storm"))
    lam_first = lam
    lo, V_lo = lam, V
    hi = None
    while hi is None:                   # march up
        trial = round(lo + step, 6)
        if trial > lam_cap:
            return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": True}
        n += 1
        V2, ok, _, _ = case.solve(trial, V_lo, max_iter, tol)
        if ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2:
            lo, V_lo = trial, V2; path.append((trial, "continuation")); continue
        n += 2
        V2, who = case.solve_any(trial, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
        if V2 is not None:
            lo, V_lo = trial, V2; path.append((trial, who)); continue
        hi = trial
    while hi - lo > width:              # bisect
        mid = round(0.5 * (lo + hi), 8)
        n += 1
        V2, ok, _, _ = case.solve(mid, V_lo, max_iter, tol)
        if not (ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2):
            n += 2
            V2, who = case.solve_any(mid, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            lo, V_lo = mid, V2; path.append((mid, "bisection"))
        else:
            hi = mid
    return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": False}


def truth_below_nose(case, lam_max, V_max, lams, tol=1e-8, max_iter=30):
    """AC solutions at the given load factors (all <= lam_max), marching down from the solution at the nose.
    Returns {lam: V} (missing where Newton failed, which should not happen below the nose)."""
    out, V = {}, V_max
    for lam in sorted(lams, reverse=True):
        V2, ok, _, _ = case.solve(lam, V, max_iter, tol)
        if not ok:
            V2, _ = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            out[lam] = V2; V = V2
    return out


def sigma_min(J, iters=40, seed=0):
    """Smallest singular value of the Jacobian, 1 / largest singular value of its inverse, by power iteration on
    J^-1 J^-T with one LU factorisation of J and one of J^T. Expected to drop toward 0 at a fold."""
    J = J.tocsc()
    lu, luT = splu(J), splu(J.T.tocsc())
    x = np.random.default_rng(seed).standard_normal(J.shape[0])
    x /= np.linalg.norm(x)
    val = np.nan
    for _ in range(iters):
        y = lu.solve(luT.solve(x))
        val = float(np.linalg.norm(y))
        x = y / val
    return 1.0 / np.sqrt(val)


# ---------------------------------------------------------------- helpers
def hop_distance(d, a, b):
    """Number of in-service lines between buses a and b (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    return float(shortest_path(A, unweighted=True, directed=False, indices=int(a))[int(b)])


def auroc(score, positive):
    """Area under the ROC curve by ranks (ties get the average rank); nan if one class is empty."""
    score, positive = np.asarray(score, float), np.asarray(positive, bool)
    ok = np.isfinite(score); score, positive = score[ok], positive[ok]
    n1, n0 = positive.sum(), (~positive).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    order = np.argsort(score, kind="mergesort")
    ranks = np.empty(len(score)); ranks[order] = np.arange(1, len(score) + 1)
    s = score[order]
    i = 0
    while i < len(s):                   # average ranks of ties
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = 0.5 * (i + j) + 1
        i = j + 1
    return float((ranks[positive].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


"""Extended verification helpers: generator reactive limits inside the AC truth and the check; stress patterns other
than the K1 scaling (loads only, reactive demand only, area-to-area transfer, generator outage on top of the storm);
warm-start timing of the classical solver. Uses LambdaCase / find_lambda_max / truth_below_nose from the load-sweep helpers and
build_ybus / newton / klu_solve / system_of / dc_angles / SN_MVA / HAVE_KLU from the shared power-flow helpers above."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import dijkstra
from scipy.sparse.linalg import splu


# ================================================================ reactive limits
def bus_q_limits(d, gen_rows):
    """Per-bus reactive limits (p.u.) from the generator table of one scenario: sums over in-service generators.
    gen_rows: DataFrame with bus, min_q_mvar, max_q_mvar, in_service, is_slack_gen. Buses without a generator get 0/0."""
    nb = d["nb"]
    qmin, qmax = np.zeros(nb), np.zeros(nb)
    g = gen_rows[gen_rows["in_service"].to_numpy() > 0.5] if "in_service" in gen_rows else gen_rows
    np.add.at(qmin, g["bus"].to_numpy().astype(int), g["min_q_mvar"].to_numpy(float) / SN_MVA)
    np.add.at(qmax, g["bus"].to_numpy().astype(int), g["max_q_mvar"].to_numpy(float) / SN_MVA)
    return qmin, qmax


def q_generation(Y, V, Qd_pu):
    """Reactive generation implied by a voltage state (p.u.): injected Q plus the reactive load."""
    return (V * np.conj(Y @ V)).imag + Qd_pu


class QLimCase(LambdaCase):
    """A LambdaCase whose solver enforces generator reactive limits: a PV bus whose generation would exceed its limit
    is held at the limit as a PQ bus (and released again if its voltage recovers past the set-point). The outer loop
    is the standard one used in power-flow programs; a case counts as solved only when no further switching is needed."""

    def __init__(self, d, qmin_pu, qmax_pu, max_outer=15, eps=1e-6):
        super().__init__(d)
        self.qmin, self.qmax, self.max_outer, self.eps = qmin_pu, qmax_pu, max_outer, eps
        self.vset = d["Vm"].copy()                      # generator set-points (the base solution holds them)
        self.limited = self.PV & (self.qmax >= self.qmin)
        self.last = {"switched": {}, "outer": 0, "iters": 0}

    def _inner(self, S, V0, pv, pq, max_iter, tol):
        if HAVE_KLU:
            return klu_solve(self.Y, S, V0, self.ref, pv, pq, max_iter, tol)
        t0 = time.perf_counter()
        V, it, norms, _ = newton(self.Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
        return V, bool(norms[-1] <= tol), it, time.perf_counter() - t0

    def solve(self, lam, V0, max_iter=30, tol=1e-8):
        """Newton-Raphson with reactive-limit switching. Returns V, converged, total inner iterations, seconds.
        `self.last` keeps the switched buses of the last call ({bus: 'max' | 'min'}) and the number of outer rounds."""
        t0 = time.perf_counter()
        S_base = self.S(lam)
        Qd = lam * self.Qd
        V = np.asarray(V0, complex).copy()
        switched, backs, total_it = {}, {}, 0
        for outer in range(1, self.max_outer + 1):
            sw = np.array(sorted(switched), dtype=int)
            pv = np.array([b for b in self.pv if b not in switched], dtype=int)
            pq = np.array(sorted(set(self.pq.tolist()) | set(switched)), dtype=int)
            S = S_base.copy()
            if len(sw):
                S[sw] = S_base.real[sw] + 1j * (np.array([self.qmax[b] if switched[b] == "max" else self.qmin[b] for b in sw]) - Qd[sw])
            V[pv] = self.vset[pv] * np.exp(1j * np.angle(V[pv]))      # PV buses start at their set-point
            V, ok, it, _ = self._inner(S, V, pv, pq, max_iter, tol)
            total_it += it
            if not (ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2):
                self.last = {"switched": dict(switched), "outer": outer, "iters": total_it}
                return V, False, total_it, time.perf_counter() - t0
            Qg = q_generation(self.Y, V, Qd)
            to_max = [b for b in pv if self.limited[b] and Qg[b] > self.qmax[b] + self.eps]
            to_min = [b for b in pv if self.limited[b] and Qg[b] < self.qmin[b] - self.eps]
            Vm = np.abs(V)
            back = [b for b in sw if backs.get(b, 0) < 2 and ((switched[b] == "max" and Vm[b] > self.vset[b] + self.eps) or
                                                             (switched[b] == "min" and Vm[b] < self.vset[b] - self.eps))]
            if not to_max and not to_min and not back:
                self.last = {"switched": dict(switched), "outer": outer, "iters": total_it}
                return V, True, total_it, time.perf_counter() - t0
            for b in to_max:
                switched[int(b)] = "max"
            for b in to_min:
                switched[int(b)] = "min"
            for b in back:
                del switched[int(b)]; backs[int(b)] = backs.get(int(b), 0) + 1
        self.last = {"switched": dict(switched), "outer": self.max_outer, "iters": total_it}
        return V, False, total_it, time.perf_counter() - t0

    def decisions(self, lam, V, margin_line=0.0, margin_v=0.0):
        """As LambdaCase.decisions, but voltage limits are checked at every non-slack bus (a generator at its limit no
        longer holds its voltage) and the reactive generation beyond the limits is reported."""
        Vm = np.abs(V)
        ld = self.loading(V)
        P, Q = self.mismatch(lam, V)
        chk = ~self.REF
        low = chk & (Vm < self.vmin + margin_v)
        high = chk & (Vm > self.vmax - margin_v)
        vm_chk = np.where(chk, Vm, np.inf)
        Qg = q_generation(self.Y, V, lam * self.Qd)
        over_q = self.limited & (Qg > self.qmax + 1e-6)
        return {"min_vm": float(vm_chk.min()), "weak_bus": int(np.argmin(vm_chk)),
                "n_low_v": int(low.sum()), "n_high_v": int(high.sum()), "n_over": int((ld >= 1.0 - margin_line).sum()),
                "max_loading": float(ld.max()) if len(ld) else np.nan,
                "resP_max_MW": float(P.max()), "resP_p99_MW": float(np.percentile(P, 99)), "resP_mean_MW": float(P.mean()),
                "resQ_max_Mvar": float(Q.max()) if len(Q) else np.nan,
                "mismatch_pu": float(max(P.max(), Q.max() if len(Q) else 0.0) / SN_MVA),
                "n_gen_over_qmax": int(over_q.sum()), "q_over_qmax_Mvar": float(SN_MVA * np.clip(Qg - self.qmax, 0, None)[self.limited].sum())}


# ================================================================ stress patterns
class StressCase(LambdaCase):
    """A storm topology stressed in one direction by a factor lambda (1 = the actual operating point):
    'all'      every load and every non-slack generator scaled together (the K1 pattern; slack takes the rest);
    'load'     loads only (P and Q at constant power factor); generation fixed, the slack takes the growth;
    'reactive' reactive demand only (Q of every load); active power unchanged;
    'transfer' loads inside an area scaled, generation outside the area scaled up by the same MW (a transfer into
               the storm area); the slack takes the losses."""

    def __init__(self, d, mode="all", area=None, kappa=None):
        super().__init__(d)
        self.mode = mode
        self.Pg, self.Pd = d["Pg"] / SN_MVA, d["Pd"] / SN_MVA
        self.area = np.zeros(self.nb, bool) if area is None else np.asarray(area, bool)
        self.genB = (self.Pg > 0) & ~self.area & ~self.REF
        self.kappa = kappa if kappa is not None else (self.Pd[self.area].sum() / max(self.Pg[self.genB].sum(), 1e-9))

    def S(self, lam):
        if self.mode == "all":
            return lam * self.P0 + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "load":
            return (self.Pg - lam * self.Pd) + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "reactive":
            return self.P0 + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "transfer":
            fl = 1.0 + (lam - 1.0) * self.area
            fg = 1.0 + (lam - 1.0) * self.kappa * self.genB
            return (self.Pg * fg - self.Pd * fl) + 1j * (self.Qg - self.Qd * fl)
        raise ValueError(self.mode)

    def dc_loading(self, lam):
        """DC loading at this lambda: angles from the scaled active injections through the DC susceptance matrix."""
        d = self.d
        nb, on = d["nb"], d["on"]
        f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
        B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
        keep = self.nonref
        th = np.zeros(nb)
        th[keep] = splu(B[keep][:, keep]).solve(self.S(lam).real[keep])
        return self.loading(np.exp(1j * (th + np.radians(float(d["Va"][self.ref[0]])))))


def storm_area(d, footprint_buses, min_load_share=0.10, max_hops=6):
    """Buses within h hops of the storm footprint, h grown until the area holds at least `min_load_share` of the load."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=np.asarray(footprint_buses, int), min_only=True)
    total = max(d["Pd"].sum(), 1e-9)
    for h in range(0, max_hops + 1):
        area = hop <= h
        if d["Pd"][area].sum() >= min_load_share * total:
            return area, h, hop
    return hop <= max_hops, max_hops, hop


def footprint_of(d, removed_rows):
    """Endpoint buses of the removed lines (branch-row indices)."""
    return np.unique(np.r_[d["f"][removed_rows], d["t"][removed_rows]])


def pick_generator_outage(d, gen_rows, hop, max_hops=3):
    """The largest in-service non-slack generator (by active output; by reactive range if none produces active power)
    whose bus lies within `max_hops` of the storm footprint; the largest anywhere if there is none. Returns its row
    label in gen_rows (or None)."""
    g = gen_rows
    keep = np.ones(len(g), bool)
    if "in_service" in g:
        keep &= g["in_service"].to_numpy(float) > 0.5
    if "is_slack_gen" in g:
        keep &= g["is_slack_gen"].to_numpy(float) < 0.5
    keep &= ~d["REF"][g["bus"].to_numpy().astype(int)]
    g = g[keep]
    if len(g) == 0:
        return None
    size = g["p_mw"].to_numpy(float)
    if not (size > 0).any():
        size = g["max_q_mvar"].to_numpy(float) - g["min_q_mvar"].to_numpy(float)
    near = hop[g["bus"].to_numpy().astype(int)] <= max_hops
    cand = g[near] if near.any() else g
    size_c = size[near] if near.any() else size
    return int(cand.index[int(np.argmax(size_c))])


def remove_generator(d, gen_rows, row):
    """The truth model after one generator is lost: its P and Q leave the bus (the slack takes the P); if no in-service
    generator remains at the bus, it becomes a PQ bus. Returns the new dict and the reduced generator table."""
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    g = gen_rows.copy()
    bus = int(g.loc[row, "bus"])
    d2["Pg"][bus] -= float(g.loc[row, "p_mw"]); d2["Qg"][bus] -= float(g.loc[row, "q_mvar"])
    g = g.drop(index=row)
    left = g[(g["bus"].to_numpy().astype(int) == bus) & (g["in_service"].to_numpy() > 0.5)] if "in_service" in g else g[g["bus"] == bus]
    if len(left) == 0 and not d2["REF"][bus]:
        d2["PV"][bus] = False; d2["PQ"][bus] = True
        d2["Pg"][bus] = 0.0; d2["Qg"][bus] = 0.0
    return d2, g, bus


# ================================================================ warm starts (item 9)
def timed_solves(case, lam, starts, max_iter=30, tol=1e-8, reps=3):
    """Iterations and the median time (ms) of the solver from each start; converged flag per start."""
    out = {}
    for name, V0 in starts:
        its, ts, oks = [], [], []
        for _ in range(reps):
            V, ok, it, dt = case.solve(lam, V0, max_iter, tol)
            its.append(it); ts.append(1000 * dt); oks.append(bool(ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2))
        out[name] = {"converged": all(oks), "iters": int(np.median(its)), "ms": float(np.median(ts))}
    return out


"""Independent-check helpers: an independent implementation of the AC power-flow equations for validation. PYPOWER (the Python port of
MATPOWER) builds its own admittance matrix from the raw branch parameters of the released tables (r, x, b, tap, shift,
status) and the bus shunts (GS, BS), its own injections from the bus loads and the generator table, and solves with its
own Newton-Raphson (and its own reactive-limit loop). Nothing of our custom Ybus (built from the stored Yff/Yft/Ytf/Ytt
entries), our Python Newton or lightsim2grid is used here, so an agreement between the two is evidence against a shared
modelling error. Uses read_cols / find_local / SN_MVA from the shared helpers above."""
import time

import numpy as np
import pandas as pd
from pypower.api import ppoption, runpf
from pypower.bustypes import bustypes
from pypower.dcpf import dcpf
from pypower.makeBdc import makeBdc
from pypower.makeSbus import makeSbus
from pypower.makeYbus import makeYbus
from pypower.newtonpf import newtonpf
from pypower.idx_bus import BUS_I, BUS_TYPE, PD, QD, GS, BS, BUS_AREA, VM, VA, BASE_KV, ZONE, VMAX, VMIN
from pypower.idx_gen import GEN_BUS, PG, QG, QMAX, QMIN, VG, MBASE, GEN_STATUS, PMAX, PMIN
from pypower.idx_brch import F_BUS, T_BUS, BR_R, BR_X, BR_B, RATE_A, RATE_B, RATE_C, TAP, SHIFT, BR_STATUS, ANGMIN, ANGMAX

RAW_BUS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "vn_kv", "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc"]
RAW_BR = ["scenario", "idx", "from_bus", "to_bus", "r", "x", "b", "tap", "shift", "rate_a", "br_status", "ang_min", "ang_max",
          "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i", "Ytt_r", "Ytt_i"]
RAW_GEN = ["scenario", "idx", "bus", "p_mw", "q_mvar", "min_p_mw", "max_p_mw", "min_q_mvar", "max_q_mvar", "in_service", "is_slack_gen"]
PP_OPT = ppoption(VERBOSE=0, OUT_ALL=0, PF_ALG=1)


RAW_CACHE = {}


def preload_raw(local_dir, level, ids):
    """Read the raw tables of many scenarios of one level at once (one pass over the parquet files) into RAW_CACHE."""
    ids = np.unique(np.asarray(ids, dtype=np.int64))
    have = RAW_CACHE.get(level)
    if have is not None:
        ids = ids[~np.isin(ids, have["ids"])]
        if len(ids) == 0:
            return
    bus = read_cols(find_local(local_dir, level, "bus_data"), RAW_BUS, scen=ids)
    br = read_cols(find_local(local_dir, level, "branch_data"), RAW_BR, scen=ids)
    gen = read_cols(find_local(local_dir, level, "gen_data"), RAW_GEN, scen=ids)
    for t in (bus, br, gen):
        t["scenario"] = t["scenario"].astype(np.int64)
    if have is None:
        RAW_CACHE[level] = {"ids": ids, "bus": bus, "br": br, "gen": gen}
    else:
        RAW_CACHE[level] = {"ids": np.r_[have["ids"], ids], "bus": pd.concat([have["bus"], bus]), "br": pd.concat([have["br"], br]), "gen": pd.concat([have["gen"], gen])}
    c = RAW_CACHE[level]
    c["g_bus"], c["g_br"], c["g_gen"] = c["bus"].groupby("scenario"), c["br"].groupby("scenario"), c["gen"].groupby("scenario")


def raw_tables(local_dir, level, sid):
    """The released bus, branch and generator tables of one scenario, with the raw network parameters."""
    sid = int(sid)
    c = RAW_CACHE.get(level)
    if c is not None and sid in c["g_bus"].groups:
        bus, br, gen = (c[g].get_group(sid).copy() for g in ["g_bus", "g_br", "g_gen"])
    else:
        bus = read_cols(find_local(local_dir, level, "bus_data"), RAW_BUS, scen=[sid])
        br = read_cols(find_local(local_dir, level, "branch_data"), RAW_BR, scen=[sid])
        gen = read_cols(find_local(local_dir, level, "gen_data"), RAW_GEN, scen=[sid])
    bus = bus.sort_values("bus").reset_index(drop=True); br = br.sort_values("idx").reset_index(drop=True); gen = gen.sort_values("idx").reset_index(drop=True)
    if len(bus) == 0 or len(br) == 0 or len(gen) == 0:
        raise RuntimeError(f"scenario {sid} of level {level} is not on disk")
    return bus, br, gen


def admittance_check(br):
    """Recompute the branch admittance entries from r, x, b, tap and shift with the MATPOWER formulas and compare with the
    stored Yff/Yft/Ytf/Ytt: a check on the units, the tap and phase-shift conventions and the branch status."""
    on = br["br_status"].to_numpy(float) > 0.5
    ys = on / (br["r"].to_numpy(float) + 1j * br["x"].to_numpy(float))
    bc = on * br["b"].to_numpy(float)
    tap = br["tap"].to_numpy(float).copy(); tap[tap == 0] = 1.0
    t = tap * np.exp(1j * np.radians(br["shift"].to_numpy(float)))
    Ytt = ys + 1j * bc / 2
    Yff = Ytt / (t * np.conj(t))
    Yft = -ys / np.conj(t)
    Ytf = -ys / t
    stored = {k: br[k + "_r"].to_numpy(float) + 1j * br[k + "_i"].to_numpy(float) for k in ["Yff", "Yft", "Ytf", "Ytt"]}
    diff = {k: float(np.max(np.abs(v - stored[k]))) for k, v in [("Yff", Yff), ("Yft", Yft), ("Ytf", Ytf), ("Ytt", Ytt)]}
    scale = float(np.max(np.abs(stored["Yff"])))
    return {"max_abs_diff": max(diff.values()), "max_rel_diff": max(diff.values()) / scale, "per_entry": diff,
            "branches": int(len(br)), "in_service": int(on.sum()), "transformers_tap_not_1": int((tap != 1.0).sum()),
            "phase_shifters": int((br["shift"].to_numpy(float) != 0).sum()), "zero_impedance": int((np.abs(br["r"] + 1j * br["x"]) == 0).sum())}


class Indep:
    """One scenario's tables as a PYPOWER case. Lines are removed by their idx; the load factor lam scales every load and
    every non-slack generator, as our LambdaCase does (the slack takes the rest); generator voltage set-points are the
    released Vm of their buses and are never changed."""

    def __init__(self, bus, br, gen):
        self.bus, self.br, self.gen = bus.reset_index(drop=True), br.reset_index(drop=True), gen.reset_index(drop=True)
        self.nb = len(self.bus)
        self.PQ, self.PV, self.REF = (self.bus[c].to_numpy(float) > 0.5 for c in ["PQ", "PV", "REF"])
        self.vmin, self.vmax = self.bus["min_vm_pu"].to_numpy(float), self.bus["max_vm_pu"].to_numpy(float)
        self.idx = self.br["idx"].to_numpy().astype(int)
        self.rate = self.br["rate_a"].to_numpy(float)

    # ---------------------------------------------------------------- the case
    def ppc(self, removed_idx=(), lam=1.0, Vm0=None, Va0=None):
        b, g, r = self.bus, self.gen, self.br
        nb, ng, nl = len(b), len(g), len(r)
        bus = np.zeros((nb, 13))
        bus[:, BUS_I] = np.arange(nb)                                   # internal numbering: our buses are already 0..nb-1
        bus[:, BUS_TYPE] = np.where(self.REF, 3, np.where(self.PV, 2, 1))
        bus[:, PD] = lam * b["Pd"].to_numpy(float); bus[:, QD] = lam * b["Qd"].to_numpy(float)
        bus[:, GS] = b["GS"].to_numpy(float) * SN_MVA; bus[:, BS] = b["BS"].to_numpy(float) * SN_MVA   # stored in p.u. (datakit divides by baseMVA)
        bus[:, BUS_AREA] = 1; bus[:, ZONE] = 1
        bus[:, VM] = b["Vm"].to_numpy(float) if Vm0 is None else np.asarray(Vm0, float)
        bus[:, VA] = b["Va"].to_numpy(float) if Va0 is None else np.asarray(Va0, float)
        bus[:, BASE_KV] = b["vn_kv"].to_numpy(float); bus[:, VMAX] = self.vmax; bus[:, VMIN] = self.vmin
        # PV/REF buses of the released tables without an in-service generator (datakit turns such buses into PQ, so the released
        # Texas data should have none; counted in the conventions check): a dummy generator holds their voltage as the flags say,
        # so that PYPOWER models the same bus types as our solver, which trusts the flags
        on = g["in_service"].to_numpy(float) > 0.5
        has = np.zeros(nb, bool); has[g["bus"].to_numpy().astype(int)[on]] = True
        dummy = np.nonzero((self.PV | self.REF) & ~has)[0]
        ng2 = ng + len(dummy)
        gen = np.zeros((ng2, 21))
        gb = np.r_[g["bus"].to_numpy().astype(int), dummy]
        gen[:, GEN_BUS] = gb
        slack = np.r_[g["is_slack_gen"].to_numpy(float) > 0.5, self.REF[dummy]]
        gen[:, PG] = np.r_[g["p_mw"].to_numpy(float), np.zeros(len(dummy))] * np.where(slack, 1.0, lam)
        gen[:, QG] = np.r_[g["q_mvar"].to_numpy(float), b["Qg"].to_numpy(float)[dummy]]
        gen[:, QMAX] = np.r_[g["max_q_mvar"].to_numpy(float), np.full(len(dummy), 1e9)]
        gen[:, QMIN] = np.r_[g["min_q_mvar"].to_numpy(float), np.full(len(dummy), -1e9)]
        gen[:, VG] = b["Vm"].to_numpy(float)[gb]                        # the set-point: the released voltage of the bus
        gen[:, MBASE] = SN_MVA
        gen[:, GEN_STATUS] = np.r_[on, np.ones(len(dummy), bool)].astype(float)
        gen[:, PMAX] = np.r_[g["max_p_mw"].to_numpy(float), np.zeros(len(dummy))]; gen[:, PMIN] = np.r_[g["min_p_mw"].to_numpy(float), np.zeros(len(dummy))]
        self.n_dummy = len(dummy)
        branch = np.zeros((nl, 17))                                     # 13 inputs + 4 flow columns that pfsoln fills
        branch[:, F_BUS] = r["from_bus"].to_numpy().astype(int); branch[:, T_BUS] = r["to_bus"].to_numpy().astype(int)
        branch[:, BR_R] = r["r"].to_numpy(float); branch[:, BR_X] = r["x"].to_numpy(float); branch[:, BR_B] = r["b"].to_numpy(float)
        branch[:, RATE_A] = self.rate
        tap = r["tap"].to_numpy(float).copy(); tap[tap == 0] = 1.0
        branch[:, TAP] = tap; branch[:, SHIFT] = r["shift"].to_numpy(float)
        status = r["br_status"].to_numpy(float) > 0.5
        if len(removed_idx):
            status = status & ~np.isin(self.idx, np.asarray(removed_idx, dtype=int))
        branch[:, BR_STATUS] = status.astype(float)
        branch[:, ANGMIN] = r["ang_min"].to_numpy(float); branch[:, ANGMAX] = r["ang_max"].to_numpy(float)
        return {"version": "2", "baseMVA": SN_MVA, "bus": bus, "gen": gen, "branch": branch}

    # ---------------------------------------------------------------- starts
    def start_state(self, ppc, start):
        """Initial complex voltage for a start: 'base' (the released state), 'flat', 'dc' (PYPOWER's DC power flow), or a
        (Vm, Va_deg) pair. Generator buses always begin at their set-point magnitude, as in PYPOWER's runpf."""
        bus, gen = ppc["bus"], ppc["gen"]
        ref, pv, pq = bustypes(bus, gen)
        if isinstance(start, str):
            if start == "base":
                Vm, Va = self.bus["Vm"].to_numpy(float), np.radians(self.bus["Va"].to_numpy(float))
            elif start == "flat":
                Vm, Va = np.ones(self.nb), np.radians(np.where(self.REF, self.bus["Va"].to_numpy(float), 0.0))
            elif start == "dc":
                Bbus, Bf, Pbusinj, Pfinj = makeBdc(ppc["baseMVA"], bus, ppc["branch"])
                Pbus = makeSbus(ppc["baseMVA"], bus, gen).real - Pbusinj - bus[:, GS] / ppc["baseMVA"]
                Va = dcpf(Bbus, Pbus, np.radians(np.where(self.REF, self.bus["Va"].to_numpy(float), 0.0)), ref, pv, pq)
                Vm = np.ones(self.nb)
            else:
                raise ValueError(start)
        else:
            Vm, Va = np.asarray(start[0], float), np.radians(np.asarray(start[1], float))
        V0 = Vm * np.exp(1j * Va)
        on = np.nonzero(gen[:, GEN_STATUS] > 0)[0]
        gbus = gen[on, GEN_BUS].astype(int)
        vcb = np.ones(self.nb); vcb[pq] = 0
        k = np.nonzero(vcb[gbus])[0]
        V0[gbus[k]] = gen[on[k], VG] / np.abs(V0[gbus[k]]) * V0[gbus[k]]
        return V0, ref, pv, pq

    # ---------------------------------------------------------------- solves
    def mismatch(self, ppc, V):
        """PYPOWER's power-flow mismatch (largest entry, p.u.) of a voltage state: P at every non-slack bus, Q at PQ buses."""
        bus, gen = ppc["bus"], ppc["gen"]
        ref, pv, pq = bustypes(bus, gen)
        Ybus, _, _ = makeYbus(ppc["baseMVA"], bus, ppc["branch"])
        Sbus = makeSbus(ppc["baseMVA"], bus, gen)
        mis = V * np.conj(Ybus @ V) - Sbus
        F = np.r_[mis[pv].real, mis[pq].real, mis[pq].imag]
        return float(np.max(np.abs(F))) if len(F) else 0.0

    def solve(self, removed_idx=(), lam=1.0, start="base", max_it=30, tol=1e-8):
        """PYPOWER's Newton-Raphson on the case. Returns a dict: converged, iterations, seconds, mismatch at the start and at
        the end (p.u.), the state (Vm, Va_deg) and the screening decisions made on it."""
        ppc = self.ppc(removed_idx, lam)
        V0, ref, pv, pq = self.start_state(ppc, start)
        opt = PP_OPT.copy(); opt["PF_TOL"], opt["PF_MAX_IT"] = tol, int(max_it)
        Ybus, Yf, Yt = makeYbus(ppc["baseMVA"], ppc["bus"], ppc["branch"])
        Sbus = makeSbus(ppc["baseMVA"], ppc["bus"], ppc["gen"])
        m0 = self.mismatch(ppc, V0)
        t0 = time.perf_counter()
        note = ""
        try:
            V, ok, it = newtonpf(Ybus, Sbus, V0, ref, pv, pq, opt)
        except Exception as exc:                      # a singular Jacobian on a diverging path makes SuperLU raise: that is a failed start
            V, ok, it = np.full(len(V0), np.nan, dtype=complex), False, int(max_it)
            note = f"solver error: {type(exc).__name__}"
        sec = time.perf_counter() - t0
        ok = bool(ok) and bool(np.all(np.isfinite(V)))
        out = {"converged": ok, "iterations": int(it), "seconds": sec, "mismatch_start_pu": m0, "note": note,
               "mismatch_end_pu": self.mismatch(ppc, V) if np.all(np.isfinite(V)) else np.inf}
        out.update(self.decisions(ppc, Yf, Yt, V, Ybus, Sbus, lam))
        return out

    def decisions(self, ppc, Yf, Yt, V, Ybus, Sbus, lam):
        if not np.all(np.isfinite(V)):
            nb = len(V)
            return {"Vm": np.full(nb, np.nan), "Va_deg": np.full(nb, np.nan), "loading": np.full(len(ppc["branch"]), np.nan), "overloaded_idx": [],
                    "n_over": 0, "max_loading": np.nan, "min_vm": np.nan, "weak_bus": -1, "n_low_v": 0, "n_high_v": 0, "low_v_buses": [], "Qg_bus_Mvar": np.full(nb, np.nan)}
        branch = ppc["branch"]
        f, t = branch[:, F_BUS].astype(int), branch[:, T_BUS].astype(int)
        Sf = V[f] * np.conj(Yf @ V) * ppc["baseMVA"]
        St = V[t] * np.conj(Yt @ V) * ppc["baseMVA"]
        rated = (branch[:, BR_STATUS] > 0) & (branch[:, RATE_A] > 0)
        ld = np.full(len(branch), np.nan); ld[rated] = np.maximum(np.abs(Sf), np.abs(St))[rated] / branch[rated, RATE_A]
        Vm = np.abs(V)
        pq_vm = np.where(self.PQ, Vm, np.inf)
        inj = V * np.conj(Ybus @ V)
        Qg_bus = (inj.imag + lam * self.bus["Qd"].to_numpy(float) / ppc["baseMVA"]) * ppc["baseMVA"]
        return {"Vm": Vm, "Va_deg": np.degrees(np.angle(V)), "loading": ld, "overloaded_idx": self.idx[rated & (ld >= 1.0)].tolist(),
                "n_over": int(np.nansum(ld >= 1.0)), "max_loading": float(np.nanmax(ld)) if rated.any() else np.nan,
                "min_vm": float(pq_vm.min()), "weak_bus": int(np.argmin(pq_vm)),
                "n_low_v": int((self.PQ & (Vm < self.vmin)).sum()), "n_high_v": int((self.PQ & (Vm > self.vmax)).sum()),
                "low_v_buses": np.nonzero(self.PQ & ((Vm < self.vmin) | (Vm > self.vmax)))[0].tolist(),
                "Qg_bus_Mvar": Qg_bus}

    def solve_qlim(self, removed_idx=(), lam=1.0, start="base", max_it=30, tol=1e-8, slack_unlimited=True, convention="matpower",
                   max_rounds=15, eps=1e-6):
        """Reactive-limit enforcement around PYPOWER's Newton-Raphson (PYPOWER's own runpf loop fails on current numpy, so the
        outer loop is written here with PYPOWER's primitives). Two conventions:
          'matpower'  as MATPOWER/PYPOWER: a generator beyond a limit is fixed at that limit and its bus becomes PQ (the other
                      generators at the bus keep the reactive output of the last solution); nothing is converted back;
          'release'   as our notebook N1: limits summed per bus; a PV bus whose total exceeds a limit is held at the sum as a
                      PQ bus, and released again when its voltage recovers past the set-point (at most twice per bus).
        slack_unlimited leaves the slack generator unlimited (as in N1); with False the slack bus is limited like the others and
        a PV bus takes over as slack when it is converted. Returns converged, rounds, the state, the decisions, the buses held at
        a limit and the generators' reactive outputs."""
        from pypower.pfsoln import pfsoln
        ppc = self.ppc(removed_idx, lam)
        bus, gen, branch = ppc["bus"].copy(), ppc["gen"].copy(), ppc["branch"].copy()
        V, ref, pv, pq = self.start_state(ppc, start)
        if slack_unlimited:
            s_ = np.r_[self.gen["is_slack_gen"].to_numpy(float) > 0.5, np.zeros(len(gen) - len(self.gen), bool)]
            gen[s_, QMAX], gen[s_, QMIN] = 1e9, -1e9
        opt = PP_OPT.copy(); opt["PF_TOL"], opt["PF_MAX_IT"] = tol, int(max_it)
        Ybus, Yf, Yt = makeYbus(ppc["baseMVA"], bus, branch)
        on = gen[:, GEN_STATUS] > 0
        gbus = gen[:, GEN_BUS].astype(int)
        qmin_bus, qmax_bus = np.zeros(self.nb), np.zeros(self.nb)
        np.add.at(qmin_bus, gbus[on], gen[on, QMIN]); np.add.at(qmax_bus, gbus[on], gen[on, QMAX])
        held = {}                       # bus -> 'max' | 'min'
        backs = {}
        t0 = time.perf_counter(); total_it = 0; ok = False; rounds = 0
        vset = gen[:, VG].copy()
        bus_types0 = bus[:, BUS_TYPE].copy()
        for rounds in range(1, max_rounds + 1):
            bus[:, BUS_TYPE] = bus_types0
            for b_ in held:
                bus[b_, BUS_TYPE] = 1
            ref, pv, pq = bustypes(bus, gen)
            if len(ref) == 0:
                break
            # fixed injections of the held buses: generation at the summed limit (the gens' QG entries sum to it)
            for b_, side in held.items():
                gi = np.nonzero(on & (gbus == b_))[0]
                lim = gen[gi, QMAX] if side == "max" else gen[gi, QMIN]
                gen[gi, QG] = lim
            Sbus = makeSbus(ppc["baseMVA"], bus, gen)
            # PV buses start at their set-point magnitude
            for b_ in pv:
                gi = np.nonzero(on & (gbus == b_))[0]
                V[b_] = vset[gi[0]] * np.exp(1j * np.angle(V[b_]))
            try:
                V, ok, it = newtonpf(Ybus, Sbus, V, ref, pv, pq, opt)
            except Exception:
                V, ok, it = np.full(len(V), np.nan, dtype=complex), False, int(max_it)
            total_it += int(it)
            if not (ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2):
                ok = False
                break
            bus_, gen_, branch_ = pfsoln(ppc["baseMVA"], bus.copy(), gen.copy(), branch.copy(), Ybus, Yf, Yt, V, ref, pv, pq)
            Qg_bus = np.zeros(self.nb); np.add.at(Qg_bus, gbus[on], gen_[on, QG])
            Vm = np.abs(V)
            if convention == "matpower":
                viol_max = on & (gen_[:, QG] > gen_[:, QMAX] + eps) & np.isin(gbus, pv)
                viol_min = on & (gen_[:, QG] < gen_[:, QMIN] - eps) & np.isin(gbus, pv)
                to_max = sorted(set(gbus[viol_max].tolist())); to_min = sorted(set(gbus[viol_min].tolist()) - set(to_max))
                back = []
            else:
                to_max = [b_ for b_ in pv if Qg_bus[b_] > qmax_bus[b_] + eps and qmax_bus[b_] >= qmin_bus[b_]]
                to_min = [b_ for b_ in pv if Qg_bus[b_] < qmin_bus[b_] - eps and qmax_bus[b_] >= qmin_bus[b_]]
                back = [b_ for b_, side in held.items() if backs.get(b_, 0) < 2 and
                        ((side == "max" and Vm[b_] > vset[np.nonzero(gbus == b_)[0][0]] + eps) or
                         (side == "min" and Vm[b_] < vset[np.nonzero(gbus == b_)[0][0]] - eps))]
            if not to_max and not to_min and not back:
                gen = gen_
                break
            ok = False
            for b_ in to_max:
                held[int(b_)] = "max"
            for b_ in to_min:
                held[int(b_)] = "min"
            for b_ in back:
                del held[int(b_)]; backs[int(b_)] = backs.get(int(b_), 0) + 1
            if convention == "matpower":
                # the generators of a converted bus that are not themselves beyond a limit keep their last reactive output
                for b_ in list(to_max) + list(to_min):
                    gi = np.nonzero(on & (gbus == b_))[0]
                    for g_ in gi:
                        if not (viol_max[g_] or viol_min[g_]):
                            gen[g_, QG] = gen_[g_, QG]
                            gen[g_, QMAX] = gen[g_, QMIN] = gen_[g_, QG]      # frozen at that value for the rest of the loop
        sec = time.perf_counter() - t0
        out = {"converged": bool(ok), "rounds": int(rounds), "inner_iterations": int(total_it), "seconds": sec}
        Sbus = makeSbus(ppc["baseMVA"], bus, gen)
        out.update(self.decisions(ppc, Yf, Yt, V, Ybus, Sbus, lam))
        out.update({"buses_at_limit": sorted(held), "held": dict(held), "Qg_gen_Mvar": gen[:, QG].copy(), "convention": convention})
        return out


def complementarity(bus_df, gen_df, Vm, Qg_bus_Mvar, eps_q=1e-3, eps_v=1e-6):
    """Reactive-limit complementarity of a state, per PV bus with limits (per-bus aggregate of its in-service generators):
    within limits -> the voltage must sit at its set-point; at Qmax -> voltage at or below the set-point; at Qmin -> at or
    above. Returns counts of buses in each condition and of violations."""
    g = gen_df[gen_df["in_service"].to_numpy(float) > 0.5]
    nb = len(bus_df)
    qmin, qmax = np.zeros(nb), np.zeros(nb)
    np.add.at(qmin, g["bus"].to_numpy().astype(int), g["min_q_mvar"].to_numpy(float))
    np.add.at(qmax, g["bus"].to_numpy().astype(int), g["max_q_mvar"].to_numpy(float))
    has = np.zeros(nb, bool); has[g["bus"].to_numpy().astype(int)] = True
    pv = (bus_df["PV"].to_numpy(float) > 0.5) & has & (qmax >= qmin)
    vset = bus_df["Vm"].to_numpy(float)
    q = np.asarray(Qg_bus_Mvar, float); v = np.asarray(Vm, float)
    inside = pv & (q < qmax - eps_q) & (q > qmin + eps_q)
    at_max = pv & (q >= qmax - eps_q); at_min = pv & (q <= qmin + eps_q)
    beyond = pv & ((q > qmax + eps_q) | (q < qmin - eps_q))
    bad = (inside & (np.abs(v - vset) > eps_v)) | (at_max & (v > vset + eps_v)) | (at_min & (v < vset - eps_v)) | beyond
    return {"pv_buses_with_limits": int(pv.sum()), "inside": int(inside.sum()), "at_qmax": int(at_max.sum()), "at_qmin": int(at_min.sum()),
            "beyond_limits": int(beyond.sum()), "complementarity_violations": int(bad.sum()), "violating_buses": np.nonzero(bad)[0].tolist()[:20]}


def same_state(Vm_a, Vm_b, tol=1e-4):
    return bool(np.max(np.abs(np.asarray(Vm_a) - np.asarray(Vm_b))) < tol)


def state_diff(Vm_a, Va_a, Vm_b, Va_b):
    dva = (np.asarray(Va_a) - np.asarray(Va_b) + 180) % 360 - 180
    return {"max_dVm_pu": float(np.max(np.abs(np.asarray(Vm_a) - np.asarray(Vm_b)))), "max_dVa_deg": float(np.max(np.abs(dva)))}


# ---------------------------------------------------------------- notebook setup
import json, math, platform, zipfile, warnings, copy
import matplotlib.pyplot as plt
from IPython.display import display
from gridfm_graphkit.datasets.globals import PD_H, QD_H, PG_H

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)

# N-2 scenarios that are hard for GENCO: not used as storm base points, so the base points are ordinary
HARD2 = set(json.loads('[21, 22, 129, 131, 133, 158, 171, 436, 552, 555, 1231, 1251, 1264, 1282, 1294, 1396, 1446, 1449, 1457, 1521, 1574, 1578, 2212, 2222, 2288, 2353, 2390, 2394, 3314, 3379, 3851, 3852, 3900, 3903, 3905, 3927, 3928, 3947, 4622, 4628, 4698, 4711, 4722, 4723, 4762, 4777, 4778, 4779, 4798, 4801, 4851, 4853, 4878, 4896, 4898, 4899, 4902, 4920, 4923, 4924, 4961, 5231, 5258, 5368, 5382, 5949, 6025, 6057, 6208, 6230, 6294, 6319, 6320, 6356, 6367, 7259, 7261, 7356, 7658, 7684, 7689, 7844, 7863, 7870, 7882, 7968, 8188, 8255, 8411, 8487, 8595]'))
RAW3_PAT = re.compile(r"(?:^|/)(bus_data|branch_data|gen_data)\.parquet/scenario_partition=(\d+)/[^/]+$")
PART = 200


def read_all(path, scen):
    dset = ds.dataset(str(path), format="parquet", partitioning="hive")
    values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
    t = dset.to_table(filter=ds.field("scenario").isin(values)).to_pandas()
    return t.drop(columns=[c for c in ["scenario_partition"] if c in t.columns])


def pick_storm(d, extra, rng):
    """`extra` more lines to remove, the nearest (in hops) to a random centre bus, keeping the grid in one piece.
    Returns branch-row indices, the storm radius in hops and the centre bus; (None, None, centre) if impossible."""
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    center = int(rng.integers(nb))
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=center)
    dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
    order = cand[np.lexsort((rng.random(len(cand)), dist))]
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None, center
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]])))
    return np.array(removed), radius, center


print("ready | lightsim2grid (KLU) available:", HAVE_KLU)


def truth_dicts(level, ids):
    """Stored AC solutions of the given scenarios as compact dicts (the truth sits in the prediction slot too)."""
    ids = np.sort(np.asarray(ids, dtype=np.int64))
    bus = read_cols(find_local(LOCAL_DIR, level, "bus_data"), ["scenario", "bus", "Vm", "Va"], scen=ids)
    pred = pd.DataFrame({"scenario": bus.scenario.astype(np.int64), "bus": bus.bus.astype(np.int64),
                         "vm": bus.Vm, "va": np.radians(bus.Va)}).sort_values(["scenario", "bus"]).reset_index(drop=True)
    A = load_chunk(find_local(LOCAL_DIR, level, "bus_data"), find_local(LOCAL_DIR, level, "branch_data"), pred, ids)
    return {int(ids[i]): extract_scenario(A, i) for i in range(len(ids))}

## 1. The frozen protocol

In [ ]:
# The frozen protocol of the final evaluation. Nothing below is changed after the results are seen.
FROZEN = {
    "model": {"repo": REPO_ID, "checkpoint": MODEL_DIR, "precision": "float32, eager (the released model as is); compiled float16 timed separately",
              "graphkit_branch": GRAPHKIT_BRANCH},
    "operating_points": {"level": LEVEL_BASE, "data_partitions": EVAL_PARTS,
                         "rule": "N-2 scenarios of the conformal test role that appear in no earlier notebook result (base point, evaluated "
                                 "scenario or hard case) and are not in the K1 base-point partitions 0-1; sorted by scenario id; the first PILOT_BASES "
                                 "are the pilot, the next N_FINAL (set by the pilot's runtime, never by its results) are the final set"},
    "contingencies": {"representative": FAMILIES, "challenge": CHALLENGE, "seed_rule": "numpy default_rng(SEED * 1000003 + base scenario id) per base point, "
                      "so that the cases of a base point do not depend on how many base points are used",
                      "construction": "N2 pick_lines: extra lines removed on top of the base point's outages, grid kept in one piece"},
    "policy": POLICY, "workflows": WORKFLOWS, "gate": GATE,
    "definitions": {"verified_safe": "converged state (largest mismatch < tol_pu) with no line loading >= 1.0 and no PQ-bus voltage outside its limits",
                    "verified_violation": "converged state with at least one such violation",
                    "unresolved": "no start converged within the iteration cap; never a pass",
                    "reference": "the verified state of the classical multi-start workflow; when it has none, the verified state of another workflow if any "
                                 "(then the case counts as resolved only with GENCO assistance, and that state is checked with PYPOWER)",
                    "missed_decision": "a line overloaded or a PQ bus outside its limits in the reference that the answer does not show",
                    "false_alarm": "an unresolved verdict on a case with a reference",
                    "primary_metrics": ["missed decisions per workflow with their denominators", "false alarms", "unresolved cases", "cases resolved only with GENCO assistance",
                                        "end-to-end time per case (median, 95th percentile) overall and on the cases the classical workflow could not solve", "results by family"],
                    "uncertainty": "cluster bootstrap over base points (BOOTSTRAP resamples, 2.5-97.5 percentiles); one-sided 95 % Clopper-Pearson with the number of base points as n for zero counts"},
    "sizing": {"pilot_bases": PILOT_BASES, "time_budget_min": TIME_BUDGET_MIN, "max_bases": MAX_BASES, "rule": "N_FINAL = min(MAX_BASES, floor(TIME_BUDGET_MIN * 60 / seconds per base point in the pilot))"},
}
json.dump(FROZEN, open(Path(OUT_DIR) / "frozen_protocol.json", "w"), indent=1, default=str)
print(json.dumps({k: FROZEN[k] for k in ["policy", "workflows", "gate", "sizing"]}, indent=1, default=str))

## 2. Which N-2 data blocks hold untouched operating points; download of the model and those blocks

In [ ]:
# Which N-2 data blocks hold untouched operating points? Only the scenario column of each block's bus table is read (remotely,
# through the Hugging Face file system; a few KB per block), so that nothing is downloaded before it is chosen.
EXCL = json.loads(EXCLUSIONS_JSON)
used, test_role = set(EXCL["used_n2_ids"]), set(EXCL["n2_test_role_ids"])
from huggingface_hub import HfApi, snapshot_download
import pyarrow.parquet as pq
all_files = [f.path for f in HfApi().list_repo_tree(REPO_ID, recursive=True, revision=HF_REVISION) if getattr(f, "size", None) is not None]
by_part = {}
for p_ in all_files:
    m = re.search(rf"^data/test_{LEVEL_BASE}/[^/]+/raw/bus_data\.parquet/scenario_partition=(\d+)/[^/]+$", p_)
    if m:
        by_part.setdefault(int(m.group(1)), []).append(p_)
print(f"{len(by_part)} N-{LEVEL_BASE} data blocks in the repository")


def scenario_ids_remote(paths):
    try:
        from huggingface_hub import HfFileSystem
        fs = HfFileSystem()
        ids = []
        for p_ in paths:
            with fs.open(f"{REPO_ID}@{HF_REVISION}/{p_}", "rb") as f_:
                ids.append(pq.read_table(f_, columns=["scenario"]).column("scenario").to_numpy().astype(np.int64))
        return np.unique(np.concatenate(ids))
    except Exception as exc:
        print(f"  remote column read failed ({type(exc).__name__}: {str(exc)[:120]}); downloading the bus table of this block instead")
        snapshot_download(REPO_ID, allow_patterns=paths, local_dir=LOCAL_DIR, revision=HF_REVISION)
        ids = [pq.read_table(Path(LOCAL_DIR) / p_, columns=["scenario"]).column("scenario").to_numpy().astype(np.int64) for p_ in paths]
        return np.unique(np.concatenate(ids))


SCAN = {}
for p_ in sorted(by_part):
    if p_ in SKIP_PARTS:
        continue
    ids = scenario_ids_remote(by_part[p_])
    untouched = [int(x) for x in ids if int(x) in test_role and int(x) not in used]
    SCAN[p_] = {"scenarios": int(len(ids)), "untouched": len(untouched)}
    print(f"  block {p_}: {len(ids)} scenarios, {len(untouched)} untouched")
    good = [q for q, v in SCAN.items() if v["untouched"] >= MIN_UNTOUCHED_PER_PART]
    if len(good) >= N_PARTS:
        break
ranked = sorted(SCAN, key=lambda q: -SCAN[q]["untouched"])
EVAL_PARTS = sorted(ranked[:N_PARTS])
BASE_PARTS = EVAL_PARTS
FROZEN["operating_points"]["data_partitions"] = EVAL_PARTS
FROZEN["operating_points"]["block_scan"] = SCAN
json.dump(FROZEN, open(Path(OUT_DIR) / "frozen_protocol.json", "w"), indent=1, default=str)
print(f"chosen data blocks: {EVAL_PARTS} with {[SCAN[q]['untouched'] for q in EVAL_PARTS]} untouched operating points")
if sum(SCAN[q]["untouched"] for q in EVAL_PARTS) < PILOT_BASES + 20:
    raise RuntimeError("too few untouched operating points in the scanned blocks; raise N_PARTS or scan more blocks")

In [ ]:
from huggingface_hub import HfApi, snapshot_download
print("Listing the repository ...")
all_files = [(f.path, f.size) for f in HfApi().list_repo_tree(REPO_ID, recursive=True, revision=HF_REVISION) if getattr(f, "size", None) is not None]
local_has = lambda p: (Path(LOCAL_DIR) / p).exists()
plan = {p: s for p, s in all_files if p.startswith(MODEL_DIR + "/")}
for p, s in all_files:
    m = RAW3_PAT.search(p) if p.startswith(f"data/test_{LEVEL_BASE}/") else None
    if m and int(m.group(2)) in BASE_PARTS:
        plan[p] = s
new = sum(s for p, s in plan.items() if not local_has(p)) / 1e9
print(f"Planned: {len(plan)} files, {new:.2f} GB not yet on disk")
if new > MAX_DOWNLOAD_GB:
    raise RuntimeError("Planned download is above MAX_DOWNLOAD_GB")
snapshot_download(REPO_ID, allow_patterns=list(plan), local_dir=LOCAL_DIR, revision=HF_REVISION)
print("Download finished")

## 3. Operating points, GENCO, and a check of the model

In [ ]:
# The untouched operating points: scenarios on disk, in the conformal test role, never used before
on_disk = scenario_ids(find_local(LOCAL_DIR, LEVEL_BASE, "bus_data"))
pool = np.array(sorted(int(s) for s in on_disk if int(s) in test_role and int(s) not in used), dtype=np.int64)
print(f"{len(on_disk)} N-{LEVEL_BASE} scenarios on disk in partitions {EVAL_PARTS}; {len(pool)} are in the conformal test role and were never used before")
PILOT_IDS = pool[:PILOT_BASES]
REST_IDS = pool[PILOT_BASES:]
BASE_IDS = pool.copy()                        # all base points are loaded; the final set is cut later by the sizing rule
BASE = truth_dicts(LEVEL_BASE, BASE_IDS)
NB = next(iter(BASE.values()))["nb"]
print(f"{len(BASE)} operating points loaded ({NB} buses); pilot {len(PILOT_IDS)}, available for the final set {len(REST_IDS)}")

In [ ]:
# [GPU] build graphs of the base points (used for the model check)

import yaml
cfg_path = os.path.join(GRAPHKIT_DIR, "scripts", "contingency", "configs", "caseTexas_eval.yaml")
with open(cfg_path) as f:
    ARGS = NestedNamespace(**yaml.safe_load(f))
stats = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "normalizer_stats.pt"), map_location="cpu", weights_only=True)
stats_net = stats[NETWORK] if NETWORK in stats else (stats if "baseMVA" in stats else next(iter(stats.values())))
NORMALIZER = load_normalizer(ARGS)
NORMALIZER.fit_from_dict(stats_net)
TRANSFORM = get_task_transforms(ARGS)
print("normalizer baseMVA:", NORMALIZER.baseMVA, "| config:", cfg_path)

BUS_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "bus_data"), BASE_IDS)
GEN_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "gen_data"), BASE_IDS)
BR_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "branch_data"), BASE_IDS)
BUS_T = merge_agg_gen_into_bus(BUS_T, GEN_T)
BG, GG, RG = BUS_T.groupby("scenario"), GEN_T.groupby("scenario"), BR_T.groupby("scenario")


def build_graph(sid, base, Pd=None, Qd=None, removed_lines=()):
    """GENCO's input graph for base point `base` with extra lines removed and (optionally) other loads."""
    b = BG.get_group(base).sort_values("bus").reset_index(drop=True).copy()
    if Pd is not None:
        b["Pd"], b["Qd"] = np.asarray(Pd, float), np.asarray(Qd, float)
    r = RG.get_group(base).copy()
    if len(removed_lines):
        r.loc[r["idx"].astype(int).isin([int(x) for x in removed_lines]), "br_status"] = 0.0
    g = build_hetero_data_for_scenario(int(sid), b, GG.get_group(base), r)
    NORMALIZER.transform(g)
    return TRANSFORM(g)


def scaled_graph(g, lam):
    """The same graph with every load and every non-slack generator scaled by lam (the slack's generation is a
    masked input, 0, and stays 0; loads and generation are divided by baseMVA, so scaling the inputs is exact)."""
    h = g.clone()
    h["bus"].x = h["bus"].x.clone(); h["gen"].x = h["gen"].x.clone()
    h["bus"].x[:, PD_H] *= lam; h["bus"].x[:, QD_H] *= lam; h["gen"].x[:, PG_H] *= lam
    return h


BASE_GRAPHS = [build_graph(int(s), int(s)) for s in BASE_IDS[:N_CHECK]]
print(f"{len(BASE_GRAPHS)} base graphs built")

In [ ]:
# [GPU] load model

TASK = get_task(ARGS, [NORMALIZER])
sd = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "best_model_state_dict.pt"), map_location="cpu")
sd = adapt_state_dict_for_model(sd, TASK)
missing_keys, unexpected_keys = TASK.load_state_dict(sd, strict=False)
print(f"checkpoint loaded: {len(sd)} tensors | missing {len(missing_keys)} | unexpected {len(unexpected_keys)}")
if missing_keys:
    raise RuntimeError(f"The checkpoint does not match the model. Missing: {missing_keys[:5]}")
MODEL = TASK.model.to(DEVICE).eval()
print("GENCO parameters:", sum(p.numel() for p in MODEL.parameters()))


def forward(model, batch):
    return model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict, batch.mask_dict)

In [ ]:
# [GPU] run GENCO on a list of graphs; check the model against the stored AC solutions of the base points

def run_genco(graphs, bs=BATCH):
    """GENCO's (Vm, Va in radians) for each graph, in order."""
    out = [None] * len(graphs)
    for i, h in enumerate(graphs):
        h["scenario_id"] = torch.tensor([i], dtype=torch.long)
    for b in DataLoader(graphs, batch_size=bs, shuffle=False):
        b = b.to(DEVICE)
        with torch.no_grad():
            o = forward(MODEL, b)["bus"].float()
        ptr = b["bus"].ptr.cpu().tolist()
        for g, i in enumerate(b["scenario_id"].view(-1).cpu().tolist()):
            out[int(i)] = (o[ptr[g]:ptr[g + 1], VM_OUT].cpu().numpy().astype(np.float64),
                           o[ptr[g]:ptr[g + 1], VA_OUT].cpu().numpy().astype(np.float64))
    return out


t0 = time.perf_counter()
BASE_OUT = run_genco(BASE_GRAPHS[:N_CHECK])
T_FWD_MS = 1000 * (time.perf_counter() - t0) / len(BASE_OUT)
errs = np.array([np.max(np.abs(vm - BASE[int(s)]["Vm"])[BASE[int(s)]["PQ"]]) for s, (vm, va) in zip(BASE_IDS[:N_CHECK], BASE_OUT)])
print(f"GENCO on {len(BASE_OUT)} base points: worst-bus voltage error median {np.median(errs):.4f} p.u., "
      f"95th percentile {np.percentile(errs, 95):.4f} p.u. (expected about 0.001 and 0.004) | {T_FWD_MS:.0f} ms per graph incl. loading")
if np.median(errs) > 0.02 and NB > 100:
    raise RuntimeError("GENCO's error on ordinary N-2 cases is far above the known level: the model did not load correctly; check the installation and the downloaded checkpoint.")

## 4. Construction of the contingencies (frozen) and the measured GENCO path

In [ ]:
# Frozen construction of the contingencies: per base point, the representative families and the challenge family
import networkx as nx
d0 = next(iter(BASE.values()))
G = nx.Graph()
for f_, t_ in zip(d0["f"][d0["on"]], d0["t"][d0["on"]]):
    G.add_edge(int(f_), int(t_))
t0 = time.perf_counter()
EB = nx.edge_betweenness_centrality(G, k=min(BETWEENNESS_SAMPLES, G.number_of_nodes()), seed=SEED, normalized=True)
LINE_BTW = np.array([EB.get((int(f_), int(t_)), EB.get((int(t_), int(f_)), 0.0)) for f_, t_ in zip(d0["f"], d0["t"])])
print(f"edge betweenness of the grid in {time.perf_counter() - t0:.0f} s")


def pick_lines(d, extra, mode, rng):
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    centre = int(rng.integers(nb))
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=centre)
    dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
    if mode == "nearest":
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    elif mode == "uniform":
        order = rng.permutation(cand)
    elif mode == "corridor":
        h = REGION_HOPS
        while (dist <= h).sum() < 3 * extra and h < 12:
            h += 1
        reg = cand[dist <= h]
        order = reg[np.lexsort((rng.random(len(reg)), -LINE_BTW[reg]))]
    else:
        raise ValueError(mode)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, np.nan, centre
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]])))
    return np.array(removed), radius, centre


def build_cases(base_ids):
    cases, skipped = [], 0
    for b_id in base_ids:
        d = BASE[int(b_id)]
        rng = np.random.default_rng(SEED * 1000003 + int(b_id))
        for j, (mode, k) in enumerate(FAMILIES + CHALLENGE):
            extra = k - int((~d["on"]).sum())
            rem, radius, centre = pick_lines(d, extra, mode, rng) if extra > 0 else (np.array([], dtype=int), 0.0, -1)
            if rem is None:
                skipped += 1; continue
            d2 = {kk: (v.copy() if isinstance(v, np.ndarray) else v) for kk, v in d.items()}
            d2["on"][rem] = False
            d2.pop("Va_dc", None)
            cases.append({"sid": int(b_id) * 10 + j, "base": int(b_id), "family": f"{mode}-N{k}", "mode": mode, "k": k, "challenge": (mode, k) in CHALLENGE,
                          "removed_lines": d["line"][rem].astype(int).tolist(), "rows": rem, "radius_hops": radius, "centre": centre, "d": d2})
    return cases, skipped

In [ ]:
# GENCO on the CPU/GPU path, timed per case (batch 1, cold call separated); then the four workflows on every case
def case_graph(c):
    return build_graph(c["sid"], c["base"], removed_lines=c["removed_lines"])


def genco_one(c):
    """GENCO on one case with the timing of every step of the path: graph construction, transfer, inference, conversion."""
    t0 = time.perf_counter()
    g = case_graph(c)
    t1 = time.perf_counter()
    g["scenario_id"] = torch.tensor([0], dtype=torch.long)
    b = next(iter(DataLoader([g], batch_size=1, shuffle=False))).to(DEVICE)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t2 = time.perf_counter()
    with torch.no_grad():
        o = forward(MODEL, b)["bus"].float()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t3 = time.perf_counter()
    vm = o[:, VM_OUT].cpu().numpy().astype(np.float64); va = np.degrees(o[:, VA_OUT].cpu().numpy().astype(np.float64))
    t4 = time.perf_counter()
    return vm, va, {"graph_ms": 1000 * (t1 - t0), "transfer_ms": 1000 * (t2 - t1), "inference_ms": 1000 * (t3 - t2), "convert_ms": 1000 * (t4 - t3),
                    "genco_total_ms": 1000 * (t4 - t0)}


def compact(r, extra=None):
    row = {"status": r.status, "start_used": r.start_used, "iterations": r.iterations, "total_ms": r.timing_ms["total_ms"],
           "newton_ms": sum(v for k, v in r.timing_ms.items() if k.startswith("newton_")), "starts_tried": len([h for h in r.history if h["available"]]),
           "used_genco": r.start_used == "genco", "genco_attempted": any(h["start"] == "genco" and h["available"] for h in r.history),
           "n_over": r.limits.get("n_over"), "n_low_v": r.limits.get("n_low_v"), "n_high_v": r.limits.get("n_high_v"), "min_vm": r.limits.get("min_vm"),
           "min_vm_all": r.limits.get("min_vm_all_buses"), "max_loading": r.limits.get("max_loading"),
           "low_voltage_flag": r.flags.get("low_voltage_branch_suspect", False), "unresolved_reason": r.unresolved_reason}
    if extra:
        row.update(extra)
    return row


def run_cases(cases, label):
    """Pass 0: GENCO once per case (timed). Then one pass per workflow over all cases, so that every workflow is timed warm
    and in the same conditions. Returns the rows, the verified states and GENCO's answers."""
    rows, states, gstates, vers = [], {}, {}, {}
    t_all = time.perf_counter()
    for i, c in enumerate(cases):
        vm, va, tg = genco_one(c)
        gstates[c["sid"]] = (vm, va, tg)
        vers[c["sid"]] = Verifier(c["d"], POLICY)
    print(f"  GENCO on {len(cases)} cases ({time.perf_counter() - t_all:.0f} s)")
    results = {c["sid"]: {} for c in cases}
    for w in WORKFLOWS:
        t_w = time.perf_counter()
        for i, c in enumerate(cases):
            vm, va, tg = gstates[c["sid"]]; ver = vers[c["sid"]]
            if w == "genco_verify":
                r = ver.run(genco_vm=vm, genco_va_deg=va, policy_name=w)
                tot = tg["genco_total_ms"] + r.timing_ms["total_ms"]
            elif w == "classical_first":
                r = ver.run(genco_vm=vm, genco_va_deg=va, policy_name=w)
                tot = r.timing_ms["total_ms"] + (tg["genco_total_ms"] if any(h["start"] == "genco" and h["available"] for h in r.history) else 0.0)
            else:
                r = ver.run(policy_name=w)
                tot = r.timing_ms["total_ms"]
            results[c["sid"]][w] = r
            rows.append(compact(r, {"sid": c["sid"], "base": c["base"], "family": c["family"], "challenge": c["challenge"], "workflow": w, "set": label,
                                    "end_to_end_ms": tot, "prepare_ms": r.timing_ms["prepare_ms"], "ordinal": i, **{k: tg[k] for k in tg}}))
        print(f"  {w} on {len(cases)} cases ({time.perf_counter() - t_w:.0f} s)")
    for i, c in enumerate(cases):
        res = results[c["sid"]]; vm, va, tg = gstates[c["sid"]]
        g = res["genco_verify"].genco
        rows.append({"sid": c["sid"], "base": c["base"], "family": c["family"], "challenge": c["challenge"], "workflow": "genco_alone", "set": label,
                     "status": "prediction", "n_over": g["n_over"], "n_low_v": g["n_low_v"], "n_high_v": g["n_high_v"], "min_vm": g["min_vm"],
                     "max_loading": g["max_loading"], "resP_p99_MW": g["resP_p99_MW"], "mismatch_pu": g["mismatch_pu"], "end_to_end_ms": tg["genco_total_ms"],
                     "ordinal": i, **{k: tg[k] for k in tg}})
        states[c["sid"]] = {w: {"state": ({"vm": np.asarray(res[w].state["vm"], np.float32), "va_deg": np.asarray(res[w].state["va_deg"], np.float32)}
                                          if (res[w].state is not None and w in ("classical", "genco_verify")) else None),
                                "limits": res[w].limits, "status": res[w].status} for w in WORKFLOWS}
        states[c["sid"]]["genco_alone"] = {"overloaded_lines": g["overloaded_lines"], "flagged_lines_margin": g["flagged_lines_margin"],
                                           "low_voltage_buses": g["low_voltage_buses"], "high_voltage_buses": g["high_voltage_buses"]}
    gst = {sid: (v[0], v[1]) for sid, v in gstates.items()}
    return pd.DataFrame(rows), states, gst


# cold calls: the very first GENCO inference and the very first solver call, timed on their own
_c0 = build_cases(PILOT_IDS[:1])[0][0]
t0 = time.perf_counter(); _vm, _va, COLD_GENCO = genco_one(_c0); COLD_GENCO["wall_ms"] = 1000 * (time.perf_counter() - t0)
t0 = time.perf_counter(); _r = Verifier(_c0["d"], POLICY).run(genco_vm=_vm, genco_va_deg=_va, policy_name="genco_verify"); COLD_SOLVER_MS = 1000 * (time.perf_counter() - t0)
print(f"cold calls: GENCO path {COLD_GENCO['wall_ms']:.0f} ms (inference {COLD_GENCO['inference_ms']:.0f} ms), first verification {COLD_SOLVER_MS:.0f} ms")

## 5. Pilot and sizing of the final set

In [ ]:
t0 = time.perf_counter()
PILOT_CASES, skipped = build_cases(PILOT_IDS)
t_build = time.perf_counter() - t0
t0 = time.perf_counter()
R_PILOT, S_PILOT, G_PILOT = run_cases(PILOT_CASES, "pilot")
t_run = time.perf_counter() - t0
sec_per_base = (t_build + t_run) / len(PILOT_IDS)
N_FINAL = int(min(MAX_BASES, len(REST_IDS), np.floor(TIME_BUDGET_MIN * 60 / sec_per_base)))
FINAL_IDS = REST_IDS[:N_FINAL]
print(f"pilot: {len(PILOT_CASES)} cases on {len(PILOT_IDS)} base points ({skipped} constructions skipped) in {t_build + t_run:.0f} s = {sec_per_base:.1f} s per base point")
print(f"sizing rule: N_FINAL = min({MAX_BASES}, {len(REST_IDS)}, floor({TIME_BUDGET_MIN} * 60 / {sec_per_base:.1f})) = {N_FINAL} base points, "
      f"{N_FINAL * (len(FAMILIES) + len(CHALLENGE))} cases at most (about {N_FINAL * sec_per_base / 60:.0f} min)")
SIZING = {"pilot_bases": int(len(PILOT_IDS)), "pilot_cases": int(len(PILOT_CASES)), "seconds_per_base": sec_per_base, "N_FINAL": N_FINAL}

## 6. The final evaluation

In [ ]:
t0 = time.perf_counter()
FINAL_CASES, skipped_f = build_cases(FINAL_IDS)
R_FINAL, S_FINAL, G_FINAL = run_cases(FINAL_CASES, "final")
print(f"final set: {len(FINAL_CASES)} cases on {len(FINAL_IDS)} base points ({skipped_f} constructions skipped) in {time.perf_counter() - t0:.0f} s")
R = pd.concat([R_PILOT, R_FINAL], ignore_index=True)
R.to_csv(Path(OUT_DIR) / "runs.csv", index=False)
STATES = {**S_PILOT, **S_FINAL}; GSTATES = {**G_PILOT, **G_FINAL}
CASES = {c["sid"]: c for c in PILOT_CASES + FINAL_CASES}
pd.DataFrame([{k: c[k] for k in ["sid", "base", "family", "mode", "k", "challenge", "removed_lines", "radius_hops", "centre"]} for c in CASES.values()]).to_csv(Path(OUT_DIR) / "cases.csv", index=False)

## 7. Reference, primary metrics, uncertainty

In [ ]:
# The reference of every case, the decisions of every workflow against it, and the primary metrics with their uncertainty
def decision_sets(entry):
    if not entry or entry.get("status") == "unresolved" or not entry.get("limits"):
        return None
    L = entry["limits"]
    return {"over": set(L["overloaded_lines"]), "low": set(L["low_voltage_buses"]), "high": set(L["high_voltage_buses"])}


REF = {}
for sid, st in STATES.items():
    ref, src = None, None
    for w in WORKFLOWS:
        ds_ = decision_sets(st[w])
        if ds_ is not None:
            ref, src = ds_, w; break
    REF[sid] = {"sets": ref, "source": src, "classical_solved": decision_sets(st["classical"]) is not None}

W = R[R.workflow != "genco_alone"].copy()
W["has_reference"] = W.sid.map(lambda s: REF[s]["sets"] is not None)
W["classical_solved"] = W.sid.map(lambda s: REF[s]["classical_solved"])
W["verified"] = W.status != "unresolved"
W["false_alarm"] = (~W.verified) & W.has_reference
W["resolved_only_with_genco"] = W.verified & ~W.classical_solved & W.used_genco


def misses_vs_ref(sid, w):
    ref = REF[sid]["sets"]; ds_ = decision_sets(STATES[sid][w])
    if ref is None or ds_ is None:
        return np.nan, np.nan
    missed = len(ref["over"] - ds_["over"]) + len(ref["low"] - ds_["low"]) + len(ref["high"] - ds_["high"])
    return missed, len(ref["over"]) + len(ref["low"]) + len(ref["high"])


mm = [misses_vs_ref(s, w) for s, w in zip(W.sid, W.workflow)]
W["missed_decisions"] = [m[0] for m in mm]; W["reference_decisions"] = [m[1] for m in mm]
W["different_solution"] = W.missed_decisions > 0
W.to_csv(Path(OUT_DIR) / "workflows.csv", index=False)


def summarise(df):
    out = {}
    for w, g in df.groupby("workflow"):
        diff = g[~g.classical_solved]
        out[w] = {"cases": len(g), "verified": int(g.verified.sum()), "unresolved": int((~g.verified).sum()), "false_alarms": int(g.false_alarm.sum()),
                  "cases_with_a_reference": int(g.has_reference.sum()), "verified_with_a_missed_decision": int(g.different_solution.sum()),
                  "missed_decisions": int(np.nansum(g.missed_decisions)), "reference_decisions": int(np.nansum(g.reference_decisions)),
                  "resolved_only_with_genco": int(g.resolved_only_with_genco.sum()), "low_voltage_flags": int(g.low_voltage_flag.sum()),
                  "ms_median": float(g.end_to_end_ms.median()), "ms_p95": float(g.end_to_end_ms.quantile(0.95)), "ms_mean": float(g.end_to_end_ms.mean()),
                  "difficult_cases": int(len(diff)), "difficult_ms_median": float(diff.end_to_end_ms.median()) if len(diff) else np.nan,
                  "difficult_ms_p95": float(diff.end_to_end_ms.quantile(0.95)) if len(diff) else np.nan}
    return pd.DataFrame(out).T


FINAL = W[W.set == "final"]
print(f"FINAL SET: {FINAL.sid.nunique()} cases on {FINAL.base.nunique()} base points; reference available for {int(FINAL[FINAL.workflow == 'classical'].has_reference.sum())} "
      f"(classical solved {int(FINAL[FINAL.workflow == 'classical'].classical_solved.sum())})")
SUM_ALL = summarise(FINAL); SUM_ALL.to_csv(Path(OUT_DIR) / "summary_by_workflow.csv")
display(SUM_ALL)
SUM_FAM = {fam: summarise(g) for fam, g in FINAL.groupby("family")}
FAM = pd.concat(SUM_FAM, names=["family", "workflow"]); FAM.to_csv(Path(OUT_DIR) / "summary_by_family.csv")
display(FAM[["cases", "verified", "unresolved", "false_alarms", "verified_with_a_missed_decision", "resolved_only_with_genco", "ms_median", "ms_p95"]])

# uncertainty: cluster bootstrap over base points
rng = np.random.default_rng(SEED + 11)
bases = FINAL.base.unique()
BOOT = {}
for w in WORKFLOWS:
    g = FINAL[FINAL.workflow == w]
    by_base = {b: g[g.base == b] for b in bases}
    stats = {"unresolved_rate": [], "false_alarm_rate": [], "resolved_only_with_genco_rate": [], "missed_decision_case_rate": []}
    for _ in range(BOOTSTRAP):
        pick = rng.choice(bases, len(bases), replace=True)
        s = pd.concat([by_base[b] for b in pick])
        stats["unresolved_rate"].append((~s.verified).mean())
        stats["false_alarm_rate"].append(s.false_alarm.sum() / max(1, s.has_reference.sum()))
        stats["resolved_only_with_genco_rate"].append(s.resolved_only_with_genco.mean())
        stats["missed_decision_case_rate"].append(s.different_solution.sum() / max(1, s.verified.sum()))
    BOOT[w] = {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for k, v in stats.items()}
n_bases = len(bases)
cp0 = 100 * (1 - 0.05 ** (1 / max(n_bases, 1)))
print(f"Cluster bootstrap over {n_bases} base points ({BOOTSTRAP} resamples), 95 % intervals:")
for w in WORKFLOWS:
    print(f"  {w:20s} unresolved {100 * BOOT[w]['unresolved_rate'][0]:.1f}-{100 * BOOT[w]['unresolved_rate'][1]:.1f} % | false alarms {100 * BOOT[w]['false_alarm_rate'][0]:.1f}-{100 * BOOT[w]['false_alarm_rate'][1]:.1f} % "
          f"| resolved only with GENCO {100 * BOOT[w]['resolved_only_with_genco_rate'][0]:.1f}-{100 * BOOT[w]['resolved_only_with_genco_rate'][1]:.1f} % "
          f"| verified cases with a missed decision {100 * BOOT[w]['missed_decision_case_rate'][0]:.2f}-{100 * BOOT[w]['missed_decision_case_rate'][1]:.2f} %")
print(f"For a zero count, the one-sided 95 % Clopper-Pearson upper bound with the number of base points as n is {cp0:.2f} % (the {FINAL.sid.nunique()} cases are not independent).")

## 8. GENCO alone against the verified reference, and the trust gate fixed in advance

In [ ]:
# What GENCO alone would have decided, against the reference, by family (prediction errors of the unverified answer)
rows = []
for sid, st in STATES.items():
    ref = REF[sid]["sets"]
    if ref is None:
        continue
    g = st["genco_alone"]
    c = CASES[sid]
    rows.append({"sid": sid, "base": c["base"], "family": c["family"], "set": "final" if sid in S_FINAL else "pilot",
                 "ref_overloads": len(ref["over"]), "ref_vviol": len(ref["low"]) + len(ref["high"]),
                 "missed_overloads": len(ref["over"] - set(g["overloaded_lines"])), "missed_overloads_with_margin": len(ref["over"] - set(g["flagged_lines_margin"])),
                 "false_overloads": len(set(g["overloaded_lines"]) - ref["over"]), "false_flags_with_margin": len(set(g["flagged_lines_margin"]) - ref["over"]),
                 "missed_vviol": len((ref["low"] - set(g["low_voltage_buses"])) | (ref["high"] - set(g["high_voltage_buses"]))),
                 "resP_p99_MW": float(R[(R.sid == sid) & (R.workflow == "genco_alone")].resP_p99_MW.iloc[0])})
GA = pd.DataFrame(rows); GA.to_csv(Path(OUT_DIR) / "genco_alone.csv", index=False)
GAF = GA[GA.set == "final"]
tab = GAF.groupby("family").agg(cases=("sid", "count"), ref_overloads=("ref_overloads", "sum"), missed_overloads=("missed_overloads", "sum"),
                                 missed_with_margin=("missed_overloads_with_margin", "sum"), false_flags_per_case=("false_flags_with_margin", "mean"),
                                 ref_vviol=("ref_vviol", "sum"), missed_vviol=("missed_vviol", "sum"),
                                 cases_with_a_miss=("missed_overloads", lambda x: int(((x > 0) | (GAF.loc[x.index, "missed_vviol"] > 0)).sum())))
GAF = GAF.assign(miss_margin=((GAF.missed_overloads_with_margin > 0) | (GAF.missed_vviol > 0)).astype(int))
tab["cases_with_a_miss_with_margin"] = GAF.groupby("family").miss_margin.sum()
tab.to_csv(Path(OUT_DIR) / "genco_alone_by_family.csv")
display(tab)
# the trust gate fixed in advance: cases GENCO's 99th-percentile active residual places below the threshold
gated = GAF[GAF.resP_p99_MW < GATE["threshold_MW"]]
GATE_RESULT = {"threshold_MW": GATE["threshold_MW"], "cases_gated": int(len(gated)), "of": int(len(GAF)),
               "gated_cases_with_a_missed_decision": int(((gated.missed_overloads > 0) | (gated.missed_vviol > 0)).sum()),
               "gated_cases_with_a_miss_with_margin": int(((gated.missed_overloads_with_margin > 0) | (gated.missed_vviol > 0)).sum())}
print(f"Trust gate (fixed in advance: 99th-percentile active residual below {GATE['threshold_MW']:.3f} MW): {GATE_RESULT['cases_gated']} of {len(GAF)} cases pass the gate; "
      f"of those, {GATE_RESULT['gated_cases_with_a_missed_decision']} contain a decision GENCO alone misses ({GATE_RESULT['gated_cases_with_a_miss_with_margin']} with the margin).")

## 9. End-to-end timing: cold and warm, components, difficult cases, throughput

In [ ]:
# End-to-end timing on the final set: cold against warm, the components of the GENCO path, the difficult cases, throughput
F_ = R[R.set == "final"]
WARM_SKIP = 3
comp = F_[(F_.workflow == "genco_verify") & (F_.ordinal >= WARM_SKIP)]
TIMING = {"hardware": {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none", "device": str(DEVICE), "cpu_count": os.cpu_count(),
                       "torch_threads": torch.get_num_threads(), "solver": "lightsim2grid KLU (C++), one thread" if HAVE_KLU else "Python Newton-Raphson",
                       "precision": "float32 eager, batch 1"},
          "cold": {"genco_path_first_call_ms": COLD_GENCO["wall_ms"], "inference_first_call_ms": COLD_GENCO["inference_ms"], "first_verification_ms": COLD_SOLVER_MS},
          "genco_path_components_ms_median": {k: float(comp[k].median()) for k in ["graph_ms", "transfer_ms", "inference_ms", "convert_ms", "genco_total_ms", "prepare_ms"]},
          "genco_path_components_ms_p95": {k: float(comp[k].quantile(0.95)) for k in ["graph_ms", "transfer_ms", "inference_ms", "convert_ms", "genco_total_ms", "prepare_ms"]},
          "workflows": {}}
for w in WORKFLOWS:
    g = F_[(F_.workflow == w) & (F_.ordinal >= WARM_SKIP)]
    diff = g[~g.sid.map(lambda s: REF[s]["classical_solved"])]
    TIMING["workflows"][w] = {"ms_median": float(g.end_to_end_ms.median()), "ms_p95": float(g.end_to_end_ms.quantile(0.95)), "ms_mean": float(g.end_to_end_ms.mean()),
                              "newton_ms_median": float(g.newton_ms.median()), "starts_tried_mean": float(g.starts_tried.mean()), "iterations_mean": float(g.iterations.mean()),
                              "difficult_cases": int(len(diff)), "difficult_ms_median": float(diff.end_to_end_ms.median()) if len(diff) else None,
                              "difficult_ms_p95": float(diff.end_to_end_ms.quantile(0.95)) if len(diff) else None,
                              "unresolved_ms_median": float(g[g.status == "unresolved"].end_to_end_ms.median()) if (g.status == "unresolved").any() else None}
# throughput of inference alone at larger batches (the same graphs), float32 and compiled float16 when available
graphs = [case_graph(CASES[s]) for s in list(S_FINAL)[:64]]
for i, g_ in enumerate(graphs):
    g_["scenario_id"] = torch.tensor([i], dtype=torch.long)


def throughput(model, bs, amp=False, reps=3):
    batches = [b.to(DEVICE) for b in DataLoader(graphs, batch_size=bs, shuffle=False)]
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=amp and DEVICE.type == "cuda"):
        for b in batches[:2]:
            forward(model, b)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0, n = time.perf_counter(), 0
        for _ in range(reps):
            for b in batches:
                forward(model, b); n += b.num_graphs
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - t0) / n


TIMING["throughput_ms_per_case"] = {f"float32_batch{bs}": throughput(MODEL, bs) for bs in THROUGHPUT_BATCHES}
try:
    torch._dynamo.reset()
    cmodel = torch.compile(MODEL, dynamic=True)
    t0 = time.perf_counter(); throughput(cmodel, 16, amp=True, reps=1); TIMING["cold"]["compile_float16_s"] = time.perf_counter() - t0
    TIMING["throughput_ms_per_case"].update({f"compiled_float16_batch{bs}": throughput(cmodel, bs, amp=True) for bs in [1] + THROUGHPUT_BATCHES})
except Exception as exc:
    TIMING["throughput_ms_per_case"]["compiled_float16"] = f"failed: {type(exc).__name__}: {str(exc)[:200]}"
json.dump(TIMING, open(Path(OUT_DIR) / "timing.json", "w"), indent=1, default=float)
print("End-to-end time per case (warm, batch 1, float32):")
for w in WORKFLOWS:
    t_ = TIMING["workflows"][w]
    print(f"  {w:20s} median {t_['ms_median']:6.1f} ms, p95 {t_['ms_p95']:6.1f} ms | difficult cases ({t_['difficult_cases']}): median {t_['difficult_ms_median']} ms, p95 {t_['difficult_ms_p95']} ms")
print("GENCO path components (median ms): " + ", ".join(f"{k} {v:.1f}" for k, v in TIMING["genco_path_components_ms_median"].items()))
print("Throughput, inference only (ms per case): " + ", ".join(f"{k} {v:.1f}" if isinstance(v, float) else f"{k} {v}" for k, v in TIMING["throughput_ms_per_case"].items()))
print(f"Cold: first GENCO call {COLD_GENCO['wall_ms']:.0f} ms, first verification {COLD_SOLVER_MS:.0f} ms" + (f", float16 compilation {TIMING['cold'].get('compile_float16_s', float('nan')):.0f} s" if "compile_float16_s" in TIMING["cold"] else ""))

## 10. Independent check of the states that carry weight (PYPOWER)

In [ ]:
# Independent check of the states that carry weight: every case resolved only with GENCO assistance, and a random sample of verified cases
from pypower.api import ppoption
preload_raw(LOCAL_DIR, LEVEL_BASE, [CASES[s]["base"] for s in STATES])
rng = np.random.default_rng(SEED + 5)
only_genco = sorted(set(W[W.resolved_only_with_genco & (W.workflow == "genco_verify")].sid))
solved_final = sorted(s for s in S_FINAL if REF[s]["classical_solved"])
sample = sorted(rng.choice(solved_final, min(PYPOWER_SAMPLE, len(solved_final)), replace=False).tolist()) if solved_final else []
rows = []
for sid in only_genco + sample:
    c = CASES[sid]; ind = Indep(*raw_tables(LOCAL_DIR, LEVEL_BASE, c["base"]))
    w = "genco_verify" if sid in only_genco else "classical"
    st = STATES[sid][w]["state"]
    if st is None:
        continue
    Vm_, Va_ = np.array(st["vm"]), np.array(st["va_deg"])
    x = ind.solve(removed_idx=c["removed_lines"], lam=1.0, start=(Vm_, Va_), max_it=10)
    comp = complementarity(ind.bus, ind.gen, x["Vm"], x["Qg_bus_Mvar"])
    row = {"sid": sid, "family": c["family"], "kind": "resolved only with GENCO" if sid in only_genco else "sample", "workflow_state": w,
           "mismatch_under_pypower_pu": ind.mismatch(ind.ppc(c["removed_lines"], 1.0), Vm_ * np.exp(1j * np.radians(Va_))),
           "pypower_iterations_from_state": x["iterations"], "pypower_max_dVm_from_state": float(np.abs(x["Vm"] - Vm_).max()),
           "n_over_pypower": x["n_over"], "n_over_ours": STATES[sid][w]["limits"]["n_over"], "n_low_v_pypower": x["n_low_v"], "n_low_v_ours": STATES[sid][w]["limits"]["n_low_v"],
           "min_vm_all_buses": float(Vm_.min()), "gens_beyond_q_limits": comp["beyond_limits"]}
    if sid in only_genco:
        for name in ["base", "flat", "dc"]:
            y = ind.solve(removed_idx=c["removed_lines"], lam=1.0, start=name, max_it=100)
            row[f"pypower_from_{name}_100it"] = y["converged"]
    rows.append(row)
PC = pd.DataFrame(rows); PC.to_csv(Path(OUT_DIR) / "pypower_checks.csv", index=False)
if len(PC):
    print(f"PYPOWER on {len(PC)} verified states ({len(only_genco)} resolved only with GENCO, {len(sample)} sampled): mismatch under its equations at most "
          f"{PC.mismatch_under_pypower_pu.max():.1e} p.u.; started there it moved the voltages by at most {PC.pypower_max_dVm_from_state.max():.1e} p.u.; "
          f"decision counts agree in {int(((PC.n_over_pypower == PC.n_over_ours) & (PC.n_low_v_pypower == PC.n_low_v_ours)).sum())} of {len(PC)}.")
    if only_genco:
        display(PC[PC.kind == "resolved only with GENCO"])

## 11. Figure, demo export, regression fixtures and tests

In [ ]:
import matplotlib.pyplot as plt
fams = [f"{m}-N{k}" for m, k in FAMILIES + CHALLENGE]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
ax = axes[0]
x = np.arange(len(fams)); wdt = 0.2
for j, w in enumerate(WORKFLOWS):
    vals = [100 * (1 - FAM.loc[(f, w), "verified"] / FAM.loc[(f, w), "cases"]) if (f, w) in FAM.index else 0 for f in fams]
    ax.bar(x + (j - 1.5) * wdt, vals, wdt, label=w)
ax.set_xticks(x); ax.set_xticklabels(fams, rotation=20); ax.set_ylabel("unresolved cases (%)"); ax.set_title("Unresolved by family and workflow"); ax.legend(fontsize=8)
ax = axes[1]
for j, (col, lab) in enumerate([("missed_overloads", "overloads missed, no margin"), ("missed_with_margin", "overloads missed, 4.54 % margin"), ("missed_vviol", "voltage violations missed")]):
    vals = [tab.loc[f, col] / max(1, tab.loc[f, "cases"]) if f in tab.index else 0 for f in fams]
    ax.bar(x + (j - 1) * 0.25, vals, 0.25, label=lab)
ax.set_xticks(x); ax.set_xticklabels(fams, rotation=20); ax.set_ylabel("missed decisions per case"); ax.set_title("GENCO alone against the verified reference"); ax.legend(fontsize=8)
ax = axes[2]
for j, w in enumerate(WORKFLOWS):
    g = F_[(F_.workflow == w) & (F_.ordinal >= WARM_SKIP)]
    ax.bar(j, g.end_to_end_ms.median(), 0.6, label=w)
    ax.errorbar(j, g.end_to_end_ms.median(), yerr=[[0], [g.end_to_end_ms.quantile(0.95) - g.end_to_end_ms.median()]], color="k", capsize=3)
ax.set_xticks(range(len(WORKFLOWS))); ax.set_xticklabels(WORKFLOWS, rotation=20); ax.set_ylabel("ms per case (median; bar to p95)"); ax.set_title("End-to-end time per case")
plt.tight_layout(); plt.savefig(Path(OUT_DIR) / "fig_final_evaluation.png", dpi=150); plt.show()

In [ ]:
# Cases for the demo, chosen by predefined rules (first in sid order within the final set), with every state the page needs;
# and the regression fixtures of the module's tests
def first(cond):
    for sid in sorted(S_FINAL):
        if cond(sid):
            return sid
    return None


def ga_row(sid):
    r = GA[GA.sid == sid]
    return r.iloc[0] if len(r) else None


def near_limit(sid):
    L = STATES[sid]["classical"]["limits"]
    if not L:
        return False
    d = CASES[sid]["d"]
    vm = np.array(STATES[sid]["classical"]["state"]["vm"])
    v_gap = np.min(np.abs(vm[d["PQ"]] - d["min_vm_pu"][d["PQ"]])) if d["PQ"].any() else 1
    return abs(L["max_loading"] - 1.0) <= 0.01 or v_gap <= 0.003


rules = {"normal": lambda s: REF[s]["classical_solved"] and ga_row(s) is not None and ga_row(s).missed_overloads == 0 and ga_row(s).missed_vviol == 0 and (ga_row(s).ref_overloads + ga_row(s).ref_vviol) >= 1,
         "misleading_corrected": lambda s: REF[s]["classical_solved"] and ga_row(s) is not None and (ga_row(s).missed_overloads + ga_row(s).missed_vviol) >= 3,
         "recovered_with_genco": lambda s: s in set(only_genco),
         "near_limit": lambda s: REF[s]["classical_solved"] and near_limit(s),
         "unresolved": lambda s: all(STATES[s][w]["status"] == "unresolved" for w in WORKFLOWS)}
DEMO = {"policy": POLICY, "tolerance_pu": POLICY["tol_pu"], "max_iterations": POLICY["max_iter"], "cases": []}


def export_case(name, sid, extra=None):
    c = CASES[sid]; d = c["d"]
    vm_g, va_g = GSTATES[sid]
    ver = Verifier(d, POLICY)
    rg = ver.run(genco_vm=vm_g, genco_va_deg=va_g, policy_name="genco_verify")
    rc = ver.run(policy_name="classical")
    chosen = rc if rc.converged else rg
    rated = ver.rated
    entry = {"name": name, "sid": int(sid), "base_scenario": int(c["base"]), "family": c["family"], "removed_lines": c["removed_lines"],
             "lines": {"idx": d["line"].astype(int).tolist(), "from": d["f"].astype(int).tolist(), "to": d["t"].astype(int).tolist(), "on": d["on"].astype(bool).tolist(),
                       "rate_mva": d["rate"].tolist()},
             "buses": {"min_vm_pu": d["min_vm_pu"].tolist(), "max_vm_pu": d["max_vm_pu"].tolist(), "pq": d["PQ"].astype(bool).tolist(), "ref": d["REF"].astype(bool).tolist(),
                       "pre_contingency_vm": BASE[c["base"]]["Vm"].tolist()},
             "genco": {"vm": rg.genco["vm"], "va_deg": rg.genco["va_deg"], "mismatch_pu": rg.genco["mismatch_pu"], "resP_p99_MW": rg.genco["resP_p99_MW"],
                       "loading": ver.loading(ver.state(vm_g, va_g)).tolist(), "overloaded_lines": rg.genco["overloaded_lines"], "flagged_lines_margin": rg.genco["flagged_lines_margin"],
                       "low_voltage_buses": rg.genco["low_voltage_buses"], "high_voltage_buses": rg.genco["high_voltage_buses"], "n_over": rg.genco["n_over"], "n_low_v": rg.genco["n_low_v"]},
             "verified": None if not chosen.converged else
                 {"status": chosen.status, "start_used": chosen.start_used, "iterations": chosen.iterations, "mismatch_end_pu": chosen.residuals["mismatch_end_pu"],
                  "vm": chosen.state["vm"], "va_deg": chosen.state["va_deg"],
                  "loading": ver.loading(np.array(chosen.state["vm"]) * np.exp(1j * np.radians(np.array(chosen.state["va_deg"])))).tolist(),
                  "overloaded_lines": chosen.limits["overloaded_lines"], "low_voltage_buses": chosen.limits["low_voltage_buses"], "high_voltage_buses": chosen.limits["high_voltage_buses"],
                  "n_over": chosen.limits["n_over"], "n_low_v": chosen.limits["n_low_v"], "min_vm": chosen.limits["min_vm"], "low_voltage_branch_suspect": chosen.flags.get("low_voltage_branch_suspect", False)},
             "workflow_genco_verify": {"status": rg.status, "start_used": rg.start_used, "iterations": rg.iterations, "unresolved_reason": rg.unresolved_reason,
                                       "history": rg.history, "timing_ms": rg.timing_ms},
             "workflow_classical": {"status": rc.status, "start_used": rc.start_used, "iterations": rc.iterations, "unresolved_reason": rc.unresolved_reason, "history": rc.history},
             "reference_available": chosen.converged, "rated_line_idx": d["line"][rated].astype(int).tolist()}
    if extra:
        entry.update(extra)
    DEMO["cases"].append(entry)
    return entry


for name, cond in rules.items():
    sid = first(cond)
    if sid is None:
        DEMO["cases"].append({"name": name, "sid": None, "note": "no case of this kind in the final set"}); print(f"{name}: none in the final set"); continue
    e = export_case(name, sid)
    print(f"{name}: case {sid} ({e['family']}), verified status {e['verified']['status'] if e['verified'] else 'unresolved'}; GENCO alone: {e['genco']['n_over']} overloads, {e['genco']['n_low_v']} low-voltage buses")
# a reactive-limit example: the first verified case on which enforcing the limits changes the decisions
preload_raw(LOCAL_DIR, LEVEL_BASE, [CASES[s]["base"] for s in sorted(S_FINAL)[:QLIM_SEARCH]])
q_example = None
for sid in sorted(S_FINAL)[:QLIM_SEARCH]:
    if not REF[sid]["classical_solved"]:
        continue
    c = CASES[sid]; bus_, br_, gen_ = raw_tables(LOCAL_DIR, LEVEL_BASE, c["base"])
    dq = case_from_tables(bus_, br_, removed_idx=c["removed_lines"], gen=gen_)
    rq = Verifier(dq, dict(POLICY, enforce_q_limits=True, voltage_buses="nonslack")).run(policy_name="classical")
    r0 = STATES[sid]["classical"]
    if rq.status != r0["status"] or (rq.converged and (rq.limits["n_over"] != r0["limits"]["n_over"] or rq.limits["n_low_v"] + rq.limits["n_high_v"] != r0["limits"]["n_low_v"] + r0["limits"]["n_high_v"])):
        q_example = (sid, rq); break
if q_example is None:
    DEMO["cases"].append({"name": "reactive_limits", "sid": None, "note": f"no decision changed by the reactive limits among the first {QLIM_SEARCH} cases"}); print("reactive_limits: none found")
else:
    sid, rq = q_example
    e = export_case("reactive_limits", sid, {"with_q_limits": {"status": rq.status, "converged": rq.converged, "note": rq.history[0]["note"] if rq.history else "",
                                                               "vm": rq.state["vm"] if rq.converged else None, "va_deg": rq.state["va_deg"] if rq.converged else None,
                                                               "n_over": rq.limits.get("n_over"), "n_low_v": rq.limits.get("n_low_v"), "n_high_v": rq.limits.get("n_high_v"),
                                                               "overloaded_lines": rq.limits.get("overloaded_lines"), "low_voltage_buses": rq.limits.get("low_voltage_buses"),
                                                               "high_voltage_buses": rq.limits.get("high_voltage_buses"), "voltage_buses_checked": "every non-slack bus"}})
    print(f"reactive_limits: case {sid}: without limits {e['verified']['status']} ({e['verified']['n_over']} overloads, {e['verified']['n_low_v']} low); with limits {rq.status} ({rq.limits.get('n_over')}, {rq.limits.get('n_low_v')})")
json.dump(DEMO, open(Path(OUT_DIR) / "demo_cases.json", "w"), default=float)
print(f"demo export: {sum(1 for c_ in DEMO['cases'] if c_.get('sid') is not None)} cases, {os.path.getsize(Path(OUT_DIR) / 'demo_cases.json') / 1e6:.1f} MB")

# regression fixtures: the recovered case (if any), an ordinary case, the reactive-limit case, a case that must stay unresolved
paths = {"bus": str(find_local(LOCAL_DIR, LEVEL_BASE, "bus_data")), "branch": str(find_local(LOCAL_DIR, LEVEL_BASE, "branch_data")), "gen": str(find_local(LOCAL_DIR, LEVEL_BASE, "gen_data"))}
FIX = {"policy": POLICY, "written_by": "notebook F1", "cases": []}
for c_ in DEMO["cases"]:
    if c_.get("sid") is None:
        continue
    if c_["name"] == "recovered_with_genco":
        FIX["cases"].append({"name": "recovered_with_genco (classical workflow must stay unresolved)", "base_scenario": c_["base_scenario"], "removed_lines": c_["removed_lines"],
                             "policy": "classical", "expected_status": "unresolved", "data_paths": paths})
    elif c_["name"] in ("normal", "near_limit") and c_["verified"]:
        FIX["cases"].append({"name": c_["name"], "base_scenario": c_["base_scenario"], "removed_lines": c_["removed_lines"], "policy": "classical",
                             "expected_status": c_["verified"]["status"], "expected_start_used": c_["workflow_classical"]["start_used"], "data_paths": paths})
    elif c_["name"] == "unresolved":
        FIX["cases"].append({"name": "unresolved under the stated policy", "base_scenario": c_["base_scenario"], "removed_lines": c_["removed_lines"], "policy": "classical_recovery",
                             "expected_status": "unresolved", "data_paths": paths})
    elif c_["name"] == "reactive_limits" and c_["verified"]:
        FIX["cases"].append({"name": "reactive_limits (without limits)", "base_scenario": c_["base_scenario"], "removed_lines": c_["removed_lines"], "policy": "classical",
                             "expected_status": c_["verified"]["status"], "data_paths": paths})
json.dump(FIX, open(Path(OUT_DIR) / "regression_fixtures.json", "w"), indent=1, default=float)
Path(OUT_DIR, "genco_verify.py").write_text(GENCO_VERIFY_SOURCE)
Path(OUT_DIR, "test_genco_verify.py").write_text(TEST_SOURCE)
r_ = subprocess.run([sys.executable, str(Path(OUT_DIR) / "test_genco_verify.py")], capture_output=True, text=True, cwd=".")
print(r_.stdout[-1500:], r_.stderr[-500:])

## 12. Summary and output files

In [ ]:
import platform
summary = {"frozen_protocol": FROZEN, "sizing": SIZING, "operating_points": {"on_disk": int(len(on_disk)), "untouched_pool": int(len(pool)), "pilot": PILOT_IDS.tolist(), "final": FINAL_IDS.tolist()},
           "graphkit": {"checked_out_commit": subprocess.run(["git", "-C", GRAPHKIT_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
                        "module_file": os.path.abspath(gridfm_graphkit.__file__)},
           "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "torch": torch.__version__, "device": str(DEVICE),
                        "lightsim2grid": HAVE_KLU, "torch_scatter_shim": bool(USED_SHIM)},
           "final_set": {"cases": int(FINAL.sid.nunique()), "base_points": int(FINAL.base.nunique()), "by_workflow": SUM_ALL.to_dict(orient="index"),
                         "by_family": {f: SUM_FAM[f].to_dict(orient="index") for f in SUM_FAM}, "bootstrap_95": BOOT, "clopper_pearson_zero_upper_bound_pct_bases": cp0},
           "genco_alone_by_family": tab.to_dict(orient="index"), "gate": GATE_RESULT, "timing": TIMING,
           "pypower_checks": {"states_checked": int(len(PC)), "max_mismatch_pu": float(PC.mismatch_under_pypower_pu.max()) if len(PC) else None,
                              "max_dVm_from_state": float(PC.pypower_max_dVm_from_state.max()) if len(PC) else None, "resolved_only_with_genco": only_genco},
           "demo_cases": [{k: v for k, v in c_.items() if k in ("name", "sid", "family", "note")} for c_ in DEMO["cases"]]}
json.dump(summary, open(Path(OUT_DIR) / "summary.json", "w"), indent=1, default=lambda o: None if (isinstance(o, float) and not np.isfinite(o)) else (o.item() if hasattr(o, "item") else str(o)))
S_ = SUM_ALL
print("VERDICT (final set, untouched; the pilot is reported separately in runs.csv)")
print(f"- {FINAL.sid.nunique()} cases on {FINAL.base.nunique()} base points: classical multi-start solved {int(S_.loc['classical', 'verified'])}, "
      f"GENCO + verify {int(S_.loc['genco_verify', 'verified'])}, classical-first-then-GENCO {int(S_.loc['classical_first', 'verified'])}, classical recovery {int(S_.loc['classical_recovery', 'verified'])}.")
print(f"- Resolved only with GENCO assistance: {int(S_.loc['genco_verify', 'resolved_only_with_genco'])} cases (classical recovery under a larger budget resolved "
      f"{int(S_.loc['classical_recovery', 'verified'] - S_.loc['classical', 'verified'])} of the classical failures).")
print(f"- Verified answers that differ from the reference (a missed decision): " + ", ".join(f"{w} {int(S_.loc[w, 'verified_with_a_missed_decision'])}" for w in WORKFLOWS)
      + f"; false alarms: " + ", ".join(f"{w} {int(S_.loc[w, 'false_alarms'])}" for w in WORKFLOWS) + ".")
print(f"- Time per case (median / p95 ms): " + "; ".join(f"{w} {S_.loc[w, 'ms_median']:.1f} / {S_.loc[w, 'ms_p95']:.1f}" for w in WORKFLOWS) + ".")
print(f"- GENCO alone misses decisions in {int(((GAF.missed_overloads > 0) | (GAF.missed_vviol > 0)).sum())} of {len(GAF)} cases with a reference "
      f"({int(((GAF.missed_overloads_with_margin > 0) | (GAF.missed_vviol > 0)).sum())} with the margin); gate: {GATE_RESULT['gated_cases_with_a_missed_decision']} misses among {GATE_RESULT['cases_gated']} gated cases.")
zip_path = Path(OUT_DIR).with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path(OUT_DIR).iterdir()):
        zf.write(p, arcname=f"{Path(OUT_DIR).name}/{p.name}")
print(f"\nSaved {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB).")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass